# Apex Sovereign V31-GSQ27B-Ultra (Version 21)
Combines:
1. **0.663 V31 + ATLAS 20% + Data-Parallel 8B LoRAs** (`omerfarukmerey/0-663-0-660-v31-atlas-20` & `mercurydust/name-that-llm-submission`)
2. **Full-Text 64-Layer `Qwen3.8-27B` GSQ3 LoRA (`gsq-two-t4`, Plain + Prompt-Centered Twin)** (`lightningv08/0-661-qwen3-8-27b-lora-spectral-ensemble`)
3. **Public Ungated Llama-3.1-8B 24-Layer Prefix (`ll8v5`)** (`lightningv08/ntl-olmo2-native24-final-inference-assets/llama31-prefix24`)
4. **6-Fold 120k-Pool Qwen2.5-14B LoRA (`q25_14v5`) + Nishant Part-Two (`q25_14`)** (`dhruvtibarewal/nlm-q25-v5` + `nishantdahal/ntl-handwriting-blend`)
5. **Restored `lora24_public` in Complementary Stack + Kirill Spectral Readout Bank (`night24_helper.py`)** (`kirilltym/ntl-spectral-style-heads-paired-4404`)
6. **V40A OOF-Calibrated Within-Family Stylometry Residual (`sibling_style.py`)** (`romantamrazov/name-that-llm-who-am-i`)

In [ ]:
import glob, os, subprocess, json, shutil, sys, time as _tc
import pandas as _pd

cfgs = sorted(glob.glob('/kaggle/input/**/config.json', recursive=True))
def mdir(*parts):
    c = [os.path.dirname(x) for x in cfgs if all(p in x.lower() for p in parts) and glob.glob(os.path.join(os.path.dirname(x), '*.safetensors'))]
    assert c, parts; return c[0]
def adir(name):
    h = glob.glob(f'/kaggle/input/**/{name}/heads.pt', recursive=True); assert h, name; return os.path.dirname(h[0])
def nmeta(name):
    h = glob.glob(f'/kaggle/input/**/{name}/meta.json', recursive=True); assert h, name
    d = os.path.dirname(h[0]); c = json.load(open(h[0]))['cfg']; return d, c

DATA = next(p for p in ['/kaggle/input/competitions/name-that-llm', '/kaggle/input/name-that-llm'] if os.path.exists(p + '/train.csv'))
FAST_COMMIT = len(_pd.read_csv(DATA + '/test.csv')) < 50
print('FAST_COMMIT', FAST_COMMIT, flush=True)

EXTRA = next(iter(glob.glob('/kaggle/input/**/nlm_matched_clean.parquet', recursive=True)))
POOL = next(iter(glob.glob('/kaggle/input/**/nlm_pool_v5.parquet', recursive=True)))
ATLAS = os.path.dirname(next(iter(glob.glob('/kaggle/input/**/atlas_bundle.json', recursive=True))))

B8, B14, B25 = mdir('/qwen-3/', '/8b-base/'), mdir('/qwen-3/', '/14b-base/'), mdir('/qwen2.5/', '/7b/')
B25_14 = mdir('/qwen2.5/', '/14b/')

# Llama-3.1-8B 24-layer prefix from public inference assets
BL_SRC = mdir('llama31-prefix24')
assert os.path.exists(os.path.join(BL_SRC, 'config.json')) and len(glob.glob(os.path.join(BL_SRC, '*.safetensors'))) >= 20, BL_SRC
if not FAST_COMMIT:
    _c = _tc.time()
    shutil.copytree(BL_SRC, '/tmp/llama-3.1/8b', dirs_exist_ok=True)
    BL = '/tmp/llama-3.1/8b'
    print('llama copied to local disk in', round(_tc.time() - _c), 's', flush=True)
else:
    BL = BL_SRC
print('llama', BL, flush=True)

# Qwen3.8-27B GSQ3 assets
GSQ_BASE_SRC = mdir('qwen38-27b-gsq3')
GSQ_FINAL = glob.glob('/kaggle/input/**/gsq120k-inference/gsq/final.safetensors', recursive=True)[0]
GSQ_META = glob.glob('/kaggle/input/**/gsq120k-inference/gsq/final.json', recursive=True)[0]
GSQ_COMP = glob.glob('/kaggle/input/**/gsq120k-inference/gsq/COMPLETE.json', recursive=True)[0]
GSQ_WHEELS = sorted(glob.glob('/kaggle/input/**/qwen38-offline-linux/*.whl', recursive=True))
assert len(GSQ_WHEELS) == 6, GSQ_WHEELS
print('GSQ 27B resolved:', GSQ_BASE_SRC, GSQ_FINAL, len(GSQ_WHEELS), flush=True)

N14B, N14BC = nmeta('lora_qwen3_14b')
Q25_14, Q25_14C = nmeta('lora_qwen25_14b')
DEB_SRC = os.path.dirname(glob.glob('/kaggle/input/**/deberta_v3_large/meta.json', recursive=True)[0])
if not FAST_COMMIT:
    shutil.copytree(DEB_SRC, '/kaggle/working/deberta_v3_large', dirs_exist_ok=True)
    DEB = '/kaggle/working/deberta_v3_large'
else:
    DEB = DEB_SRC

AQ8V5 = [adir(f'q8v5_f{k}') for k in range(5)] + [adir('q8v5')]
AQ14V5 = [adir(f'q14v5_f{k}') for k in range(5)] + [adir('q14v5')]
ALL8V5 = [adir(f'll8v5_f{k}') for k in range(5)] + [adir('ll8v5')]
AQ25_14V5 = [adir(f'q25_14v5_f{k}') for k in range(5)] + [adir('q25_14v5')]
print('All adapters resolved:', len(AQ8V5), len(AQ14V5), len(ALL8V5), len(AQ25_14V5), flush=True)
os.makedirs('/kaggle/working/gsq', exist_ok=True)

In [ ]:
%%writefile gauss.py
"""Shared-prompt Gaussian group head (ported, Apache-2.0, from public Kaggle notebooks by prvsiyan / kirilltym / sangam21).
Two-way model x = class mean + prompt effect + noise; exact (n<=3) or cavity-EM (n>3) injective group marginals."""
import itertools
import numpy as np
K = 18
DEFAULT = {"red": "wpca", "d": 256, "se_shrink": "lw", "sp_shrink": "lw", "lam_floor": 1e-4, "n_exact": 3, "inj": True,
           "iters": 5, "repel": True, "sp_scale": 1.0}

# =================================================================== primitives ====
def _lse(A, axis=None, keepdims=False):
    m = np.max(A, axis=axis, keepdims=True); m = np.where(np.isfinite(m), m, 0.0)
    out = np.log(np.sum(np.exp(A - m), axis=axis, keepdims=True)) + m
    if keepdims: return out
    return np.squeeze(out, axis=axis) if axis is not None else float(out.item())

def log_softmax(L): return L - _lse(L, axis=1, keepdims=True)

def _lw_shrinkage(X):
    """Ledoit-Wolf shrinkage coefficient for rows X that are already centred (sklearn formula, assume_centered)."""
    from sklearn.covariance import ledoit_wolf_shrinkage
    return float(ledoit_wolf_shrinkage(X, assume_centered=True))

def _shrunk(S, a):
    d = S.shape[0]; return (1 - a) * S + a * (np.trace(S) / d) * np.eye(d)

def _two_way(Z, y, pinv, G, iters=4):
    """additive fit Z = mu[y] + P[prompt] + E by alternating means (balanced design -> converges in 1-2 rounds)."""
    P = np.zeros((G, Z.shape[1])); cnt_p = np.bincount(pinv, minlength=G).astype(float)
    cls = np.unique(y); mu = np.zeros((K, Z.shape[1]))
    for _ in range(iters):
        R = Z - P[pinv]
        for c in cls: mu[c] = R[y == c].mean(0)
        R = Z - mu[y]; P = np.zeros_like(P); np.add.at(P, pinv, R); P /= cnt_p[:, None]
    return mu, P, cnt_p

def fit(X, y, prompts, cfg=None):
    """fit the shared-prompt Gaussian on training rows. X (N, D) raw features, y (N,) labels in 0..17, prompts (N,) group ids.
    Returns a params dict (numpy arrays + floats); temperatures default to 1 (set by the caller's cross-fit)."""
    c = dict(DEFAULT); c.update(cfg or {})
    X = np.asarray(X, dtype=np.float64); y = np.asarray(y); N, D = X.shape
    pu, pinv = np.unique(np.asarray(prompts), return_inverse=True); G = len(pu)
    mx = X.mean(0); sx = X.std(0); sx = np.where(sx > 1e-8, sx, 1.0); Xs = (X - mx) / sx
    if c["red"] == "none":
        V = np.eye(D)
    else:
        if c["red"] == "wpca":
            Pm = np.zeros((G, D)); np.add.at(Pm, pinv, Xs); Pm /= np.bincount(pinv)[:, None]; R = Xs - Pm[pinv]
        elif c["red"] == "pca": R = Xs
        else: raise ValueError(c["red"])
        d = int(min(c["d"], N - G - 1, D))
        # economy SVD through the (N x N) Gram matrix (N << D for the real views)
        if N < D:
            w, U = np.linalg.eigh(R @ R.T); o = np.argsort(w)[::-1][:d]; w, U = np.clip(w[o], 1e-12, None), U[:, o]
            V = (R.T @ U) / np.sqrt(w)
        else:
            _, _, Vt = np.linalg.svd(R, full_matrices=False); V = Vt[:d].T
    Z = Xs @ V; d = Z.shape[1]
    mu, P, ng = _two_way(Z, y, pinv, G)
    E = Z - mu[y] - P[pinv]; dof = max(N - G - len(np.unique(y)) + 1, 1)
    Se_raw = E.T @ E / dof
    ae = _lw_shrinkage(E) if c["se_shrink"] == "lw" else float(c["se_shrink"]); Se = _shrunk(Se_raw, ae)
    Pc = P - P.mean(0); Sp_raw = Pc.T @ Pc / max(G - 1, 1) - np.mean(1.0 / ng) * Se_raw
    ap = _lw_shrinkage(Pc) if c["sp_shrink"] == "lw" else float(c["sp_shrink"])
    tr = max(np.trace(Sp_raw), 1e-8 * np.trace(Se)); Sp = (1 - ap) * Sp_raw + ap * (tr / d) * np.eye(d); Sp *= c["sp_scale"]
    # joint diagonalisation: A Se A' = I, A Sp A' = diag(lam)
    L = np.linalg.cholesky(Se); Li = np.linalg.inv(L); M = Li @ Sp @ Li.T; M = 0.5 * (M + M.T)
    lam, Q = np.linalg.eigh(M); lam = np.clip(lam, c["lam_floor"], None); A = Q.T @ Li
    W = V @ A.T
    return {"mx": mx, "sx": sx, "W": W, "m": mu @ A.T, "lam": lam, "T1": 1.0, "Tg": 1.0,
            "info": {"d": int(d), "a_e": round(ae, 4), "a_p": round(ap, 4), "lam_pct": np.round(np.percentile(lam, [5, 50, 95, 100]), 3).tolist()}, "cfg": c}

def transform(X, prm): return ((np.asarray(X, dtype=np.float64) - prm["mx"]) / prm["sx"]) @ prm["W"]

_MASK = {}
def _inj_mask(n, C):
    if (n, C) not in _MASK:
        grids = np.meshgrid(*[np.arange(C)] * n, indexing="ij"); bad = np.zeros([C] * n, bool)
        for i, j in itertools.combinations(range(n), 2): bad |= grids[i] == grids[j]
        _MASK[(n, C)] = bad
    return _MASK[(n, C)]

def singleton_ll(U, m, lam):
    """(n, C) log N(u; m_c, I + diag lam) up to a constant (p integrated out)."""
    R = U[:, None, :] - m[None]; return -0.5 * (R ** 2 / (1 + lam)).sum(-1)

def joint_terms(U, m, lam):
    """a (n, C) and B (n, n, C, C) of the exact joint log-likelihood of one group (constant in c dropped)."""
    n = len(U); gam = lam / (1 + n * lam); R = U[:, None, :] - m[None]
    a = -0.5 * (R ** 2 * (1 - gam)).sum(-1); G = R * np.sqrt(gam); B = np.einsum("icd,jed->ijce", G, G)
    return a, B

def joint_table(a, B):
    """full tuple table Q[c_1, ..., c_n] = sum_i a_i + sum_{i<j} B_ij."""
    n, C = a.shape; Q = np.zeros([C] * n)
    for i in range(n):
        sh = [1] * n; sh[i] = C; Q = Q + a[i].reshape(sh)
    for i, j in itertools.combinations(range(n), 2):
        sh = [1] * n; sh[i] = C; sh[j] = C; Q = Q + B[i, j].reshape(sh)
    return Q

def exact_marginals(Q, T=1.0, inj=True):
    """per-passage log marginal posteriors (n, C) from a tuple table Q (unnormalised log joint), tempered by 1/T."""
    n = Q.ndim; C = Q.shape[0]; Qt = Q / T
    if inj and n >= 2: Qt = np.where(_inj_mask(n, C), -np.inf, Qt)
    lz = _lse(Qt); out = np.empty((n, C))
    for i in range(n):
        ax = tuple(k for k in range(n) if k != i); out[i] = (_lse(Qt, axis=ax) if ax else Qt) - lz
    return out

def em_marginals(U, m, lam, T=1.0, iters=5, repel=False, q0=None):
    """cavity EM for one group (any n >= 2): returns (n, C) log q."""
    n = len(U); v = lam / (1 + (n - 1) * lam)
    L = singleton_ll(U, m, lam) / T if q0 is None else np.log(np.clip(q0, 1e-300, None))
    q = np.exp(log_softmax(L))
    for _ in range(iters):
        s = U - q @ m; S = s.sum(0); Pm = (S[None] - s) * v                  # cavity prompt-effect means (n, d)
        R = U[:, None, :] - Pm[:, None, :] - m[None]; L = -0.5 * (R ** 2 / (1 + v)).sum(-1) / T
        if repel:   # mean-field at-most-once: discourage classes the other members already hold
            lo = np.log(np.clip(1 - q, 1e-6, None)); L = L + (lo.sum(0)[None] - lo)
        q = np.exp(log_softmax(L))
    return log_softmax(L)

LAM_DIFF = 1e6
def lam_g(prm, cfg=None):
    """lam used for groups: the fitted one, or ~infinity in the paired-difference ablation (cfg 'diff'): then the joint likelihood
    only sees within-group differences (x_i - x_j around mu_a - mu_b with covariance 2 S_e; triples: centred observations vs
    centred prototypes) and the prompt term cancels exactly."""
    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {})
    return np.full_like(prm["lam"], LAM_DIFF) if c.get("diff") else prm["lam"]

def group_logpost(U, prm, T1=None, Tg=None, cfg=None):
    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {}); n = len(U); lg = lam_g(prm, cfg)
    T1 = prm["T1"] if T1 is None else T1; Tg = prm["Tg"] if Tg is None else Tg
    if n == 1: return log_softmax(singleton_ll(U, prm["m"], prm["lam"]) / T1)
    if n <= c["n_exact"]:
        a, B = joint_terms(U, prm["m"], lg); return exact_marginals(joint_table(a, B), Tg, c["inj"])
    return em_marginals(U, prm["m"], lg, Tg, c["iters"], c["repel"])

def scores(X_pool, lab, prm, T1=None, Tg=None, cfg=None, U=None):
    """(n, 18) log-probabilities for every pool row; lab = detected group labels of the pool rows (singletons allowed)."""
    U = transform(X_pool, prm) if U is None else U; lab = np.asarray(lab); n = len(U); out = np.empty((n, prm["m"].shape[0]))
    sizes = np.bincount(lab); s1 = sizes[lab] == 1
    T1 = prm["T1"] if T1 is None else T1
    if s1.any(): out[s1] = log_softmax(singleton_ll(U[s1], prm["m"], prm["lam"]) / T1)
    for g in np.where(sizes >= 2)[0]:
        ii = np.where(lab == g)[0]; out[ii] = group_logpost(U[ii], prm, T1, Tg, cfg)
    return out

In [ ]:
%%writefile sibling_style.py
import re
import numpy as np
from functools import lru_cache
MARKERS = [',','.', ';', ':', '!', '?','—','–','-','’','‘','“','”','"',"'",'(',')','[',']','{','}','*','`','•','…','/','\\',':','\n','\n\n','\n-','\n1.','\n2.','##','```','**','> ','\t',' = ', ' e.g.', ' i.e.', 'However,','Overall,','In conclusion','Therefore,','Moreover,','First,','Finally,',' - ',': ', ' ; ', '  ']
WORDS = ['the','and','but','that','which','with','into','from','for','you','your','our','we','I','it','its','they','their','there','this','these','those','however','therefore','moreover','although','because','perhaps','might','would','could','should','like','such','just','very','also','not','if','or','as','so','by','of','to','in','on','at','is','are','be','was','were','have','has','had','may','must','all','any','each','every','many','some','more','most','only','even','still','then','when','where','why','how','what','who']
WRE=re.compile(r"[A-Za-z]+(?:[’'][A-Za-z]+)?")
SRE=re.compile(r'(?<=[.?!])\s+')
@lru_cache(maxsize=15000)
def row(t):
    t = str(t)
    s = WRE.findall(t); lo=[z.lower() for z in s]; n=max(len(s),1); nt=max(len(t),1); ns=max(len(t.splitlines()),1)
    lens=np.array([len(z) for z in s] or [0]); sent_lens=np.array([len(WRE.findall(z)) for z in SRE.split(t) if z.strip()] or [0])
    base = [np.log1p(n),np.log1p(nt), nt/n,ns/n,len(set(lo))/n, float(np.mean(lens)), float(np.std(lens)), float(np.mean(sent_lens)), float(np.std(sent_lens)),float(np.max(sent_lens)),float(np.mean(lens>=8)),sum(c.isupper() for c in t)/nt, sum(c.isdigit() for c in t)/nt,sum(ord(c)>127 for c in t)/nt, sum(c.isspace() for c in t)/nt, sum(c=='\n' for c in t)/nt]
    base += [min(t.count(m)/n,3) for m in MARKERS]
    cnt = {}
    for w in lo: cnt[w]=cnt.get(w,0)+1
    base += [cnt.get(w,0)/n for w in WORDS]
    return np.array(base, dtype=np.float32)
def features(texts):
    return np.stack([row(str(t)) for t in texts])


def apply_grouped_style_residual(logits, train_text, test_text, labels, y, groups, max_alpha=0.08, diagnostics_path=None):
    """OOF-estimated, conservative sibling re-ranking. Cross-family probability mass is unchanged."""
    import json
    import numpy as np
    from scipy.special import logsumexp, softmax
    from sklearn.model_selection import GroupKFold
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import make_pipeline
    from sklearn.metrics import balanced_accuracy_score
    Xt = features(train_text)
    Xq = features(test_text)
    out = np.asarray(logits, dtype=np.float64).copy()
    lab = np.asarray(labels)
    yy = np.asarray(y)
    g = np.asarray(groups)
    diag = {}
    for family, C in [('claude', 0.003), ('gemini', 0.003), ('gpt', 0.003), ('glm', 0.03)]:
        cols = np.flatnonzero(np.char.startswith(lab.astype(str), family))
        if len(cols) < 2:
            continue
        train_mask = np.isin(yy, cols)
        inds = np.flatnonzero(train_mask)
        X = Xt[inds]
        target = np.searchsorted(cols, yy[inds])
        grp = g[inds]
        oof = np.empty(len(target), dtype=np.int32)
        for a, b in GroupKFold(n_splits=5).split(X, target, groups=grp):
            model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=400, class_weight='balanced'))
            model.fit(X[a], target[a])
            oof[b] = model.predict(X[b])
        bacc = float(balanced_accuracy_score(target, oof))
        chance = 1.0 / len(cols)
        reliability = np.clip((bacc - chance) / 0.15, 0.0, 1.0)
        if reliability <= 0:
            diag[family] = dict(oof_balanced_accuracy=bacc, random_level=chance, alpha_mean=0.)
            continue
        model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=400, class_weight='balanced'))
        model.fit(X, target)
        style_p = np.maximum(model.predict_proba(Xq), 1e-8)
        old_logits = out[:, cols].copy()
        old_p = softmax(old_logits, axis=1)
        margin = np.partition(old_p, -2, axis=1)[:, -1] - np.partition(old_p, -2, axis=1)[:, -2]
        # Only modestly modify disputed sibling identities; preserve total provider mass.
        gate = np.clip((0.65 - margin) / 0.65, 0., 1.)
        strength = (max_alpha * reliability) * gate
        mix = (1. - strength[:, None]) * old_p + strength[:, None] * style_p
        out[:, cols] = logsumexp(old_logits, axis=1, keepdims=True) + np.log(np.maximum(mix, 1e-12))
        assert np.allclose(logsumexp(out[:, cols], axis=1), logsumexp(old_logits, axis=1), atol=1e-8)
        diag[family] = dict(oof_balanced_accuracy=bacc, random_level=chance,
                            alpha_mean=float(strength.mean()), alpha_max=float(strength.max()),
                            changed_within_family=int(np.sum(old_p.argmax(1) != mix.argmax(1))))
    if diagnostics_path:
        with open(diagnostics_path, 'w') as f: json.dump(diag, f, indent=2)
    return out

In [ ]:
%%writefile night24_helper.py
"""Fork addition: Kirill Tymchishin's spectral readout heads (dataset kirilltym/ntl-spectral-style-heads-paired-4404, bank
`synthetic4404`, mixed at 10% after ATLAS by the public 0.639-0.642 notebooks) on the raw lora24_public test features that
the complementary stack saved. CPU only.  Usage: python night24_helper.py <raw.npy | random> <test.csv> <out.npz>"""
import glob, json, os, sys
os.environ['CUDA_VISIBLE_DEVICES'] = ''
import joblib, numpy as np, pandas as pd, sklearn

bank_path = glob.glob('/kaggle/input/**/night24_readout_banks/synthetic4404.joblib', recursive=True)[0]
sys.path.insert(0, os.path.dirname(bank_path))
from readouts import predict_bank, LAYERS

raw, test_csv, out = sys.argv[1:4]
test = pd.read_csv(test_csv)
texts = test['text'].fillna('').astype(str).tolist()
audit = json.load(open(os.path.join(os.path.dirname(bank_path), 'COMPLETE.json')))['banks']['synthetic4404']
checkpoint = json.load(open(glob.glob('/kaggle/input/**/lora24/final/COMPLETE.json', recursive=True)[0]))
bank = joblib.load(bank_path)
assert bank['adapter_sha256'] == audit['adapter_sha256'] == checkpoint['files']['adapter/adapter_model.safetensors']
assert bank['layers'] == LAYERS and len(bank['labels']) == 18
if raw == 'random':   # preview: run every head of the bank on this image's scikit-learn
    z = np.random.default_rng(0).standard_normal((len(texts), 32768)).astype(np.float32)
else:
    z = np.load(raw, mmap_mode='r')
assert z.shape == (len(texts), 32768), z.shape
p, _ = predict_bank(bank, z, texts)
assert np.isfinite(p).all() and np.allclose(p.sum(1), 1, atol=1e-6)
np.savez_compressed(out, ids=test['id'].astype(str).to_numpy(dtype=str), labels=np.asarray(bank['labels'], dtype=str), probabilities=p)
print(f'spectral readout heads: {len(p)} rows, scikit-learn {sklearn.__version__}, mean top probability {p.max(1).mean():.3f}', flush=True)

In [ ]:
%%writefile /kaggle/working/gsq/gsq_infer_model.py
"""Full64-layer GSQ inference on one T4; exact base weights, FP16 compute."""
import json
import math
import struct
import time
from pathlib import Path
import numpy as np
import torch
from torch import nn
from accelerate.utils import set_module_tensor_to_device
from transformers import AutoTokenizer
from transformers.models.qwen3_5.configuration_qwen3_5 import Qwen3_5TextConfig
from transformers.models.qwen3_5 import modeling_qwen3_5 as qm
from gsq_infer_layers import GSQLinear,DenseFrozenLinear
from delta_turing import triangular_chunk_gated_delta_rule

REVISION="b5ce0b76f60020a875dee4f6ec9d934cca4121e4"


class TensorReader:
    def __init__(self,base):
        self.base=Path(base)
        self.index=json.loads((self.base/"model.safetensors.index.json").read_text())["weight_map"]
        self.headers,self.maps={},{}
        for shard in sorted(set(self.index.values())):
            path=self.base/shard
            assert path.is_file(),f"Missing exact GSQ shard: {path}"
            with path.open("rb") as stream:
                size=struct.unpack("<Q",stream.read(8))[0];assert size<2**21
                self.headers[shard]=(8+size,json.loads(stream.read(size)))
            self.maps[shard]=np.memmap(path,dtype=np.uint8,mode="r")
        missing=[key for key,shard in self.index.items() if key not in self.headers[shard][1]]
        assert all(key.startswith("mtp.") for key in missing),missing
        self.stale_mtp_keys=missing

    def has(self,key):
        return key in self.index and key in self.headers[self.index[key]][1]

    def array(self,key):
        assert self.has(key),f"Required actual GSQ tensor is missing: {key}"
        shard=self.index[key];offset,header=self.headers[shard];item=header[key]
        lo,hi=item["data_offsets"]
        dtype={"I32":"<i4","I64":"<i8","BF16":"<u2","F16":"<f2","F32":"<f4"}[item["dtype"]]
        value=self.maps[shard][offset+lo:offset+hi].view(dtype).reshape(item["shape"])
        return value,item["dtype"]

    def tensor(self,key,device="cuda",bf16_dtype=None):
        value,dtype=self.array(key)
        # A bounded one-tensor CPU copy avoids writable-buffer warnings and
        # ensures no mutation can alter the read-only safetensors mapping.
        if dtype=="BF16":
            assert bf16_dtype in (torch.float16,torch.float32)
            # BF16 -> FP32 is exact: shift the original 16 bits into the high
            # half of IEEE FP32 on CPU. T4 never sees a BF16 conversion.
            fp32=(value.astype(np.uint32)<<16).view(np.float32)
            result=torch.from_numpy(fp32).to(bf16_dtype)
        else:
            result=torch.from_numpy(value.copy())
        return result.to(device)


class QuantizedCPUEmbedding(nn.Module):
    def __init__(self,reader):
        super().__init__()
        prefix="model.language_model.embed_tokens."
        self.packed,dtype=reader.array(prefix+"weight_packed");assert dtype=="I32"
        self.scale_bits,dtype=reader.array(prefix+"weight_scale");assert dtype=="BF16"
        assert self.packed.shape==(248320,640) and self.scale_bits.shape==(248320,80)
        self.cpu_packed_bytes=self.packed.nbytes+self.scale_bits.nbytes
        self.last_lookup={}

    @torch.no_grad()
    def forward(self,ids):
        began=time.perf_counter()
        assert ids.device.type=="cpu"
        flat=ids.numpy().reshape(-1)
        unique,inverse=np.unique(flat,return_inverse=True)
        # Row selection occurs BEFORE unpacking. Never materialize all248320
        # decoded embedding rows or move the full packed table to GPU.
        words=self.packed[unique].astype(np.uint32,copy=False)
        shifts=np.arange(8,dtype=np.uint32)*4
        q=((words[:,:,None]>>shifts)&15).reshape(len(unique),5120).astype(np.int16)-8
        scales=(self.scale_bits[unique].astype(np.uint32)<<16).view(np.float32)
        values=(q.reshape(len(unique),80,64).astype(np.float32)*scales[:,:,None]).reshape(len(unique),5120)
        decoded=torch.from_numpy(values).to(torch.float16)
        selected=decoded.index_select(0,torch.from_numpy(inverse)).reshape(*ids.shape,5120)
        result=selected.to("cuda",non_blocking=False)
        self.last_lookup=dict(unique_rows=len(unique),requested_rows=len(flat),
            transferred_bytes=selected.numel()*selected.element_size(),seconds=time.perf_counter()-began)
        return result


class GSQClassifier(nn.Module):
    def __init__(self,base,*,pad_multiple=8):
        super().__init__();began=time.perf_counter()
        assert torch.cuda.device_count()==1
        self.pad_multiple=pad_multiple
        self.depths=[8,16,24,32,40,48,56,64]
        config=json.loads((Path(base)/"config.json").read_text())
        quant=config["quantization_config"]
        assert quant["quant_method"]=="compressed-tensors" and quant["format"]=="pack-quantized"
        assert quant["config_groups"]["group_0"]["weights"]["num_bits"]==3
        assert quant["config_groups"]["group_0"]["weights"]["group_size"]==128
        cfg=Qwen3_5TextConfig(**config["text_config"])
        assert cfg.num_hidden_layers==64 and cfg.hidden_size==5120
        cfg._attn_implementation="sdpa";cfg.use_cache=False
        self.cfg=cfg
        delta=triangular_chunk_gated_delta_rule
        qm.FusedRMSNormGated=None
        with torch.device("meta"):self.model=qm.Qwen3_5TextModel(cfg)
        self.model.norm=nn.Identity()
        self.reader=TensorReader(base)
        prefix="model.language_model."
        packed_count=dense_count=packed_gpu_bytes=0
        for name,module in list(self.model.named_modules()):
            if not isinstance(module,nn.Linear):continue
            assert name.startswith("layers.") and module.bias is None
            layer=int(name.split(".")[1]);rank=16 if layer>=56 else 0
            key=prefix+name+"."
            if self.reader.has(key+"weight_packed"):
                shape,_=self.reader.array(key+"weight_shape")
                assert tuple(shape)==(module.out_features,module.in_features)
                packed=self.reader.tensor(key+"weight_packed")
                scales=self.reader.tensor(key+"weight_scale",bf16_dtype=torch.float32)
                replacement=GSQLinear(packed,scales,module.in_features,module.out_features,rank)
                packed_count+=1;packed_gpu_bytes+=packed.numel()*packed.element_size()+scales.numel()*scales.element_size()
            else:
                assert ".linear_attn.in_proj_a" in name or ".linear_attn.in_proj_b" in name,name
                replacement=DenseFrozenLinear(self.reader.tensor(key+"weight",bf16_dtype=torch.float16),module.in_features,module.out_features,rank)
                dense_count+=1
            parent,_,attribute=name.rpartition(".");setattr(self.model.get_submodule(parent),attribute,replacement)
        self.model.embed_tokens=QuantizedCPUEmbedding(self.reader)
        self.model.rotary_emb=qm.Qwen3_5TextRotaryEmbedding(cfg,device="cuda")
        for name,parameter in list(self.model.named_parameters()):
            if ".lora_A" in name or ".lora_B" in name:continue
            value=self.reader.tensor(prefix+name,bf16_dtype=torch.float16)
            set_module_tensor_to_device(self.model,name,"cuda",value=value,dtype=value.dtype)
        for name,parameter in self.model.named_parameters():parameter.requires_grad_(".lora_A" in name or ".lora_B" in name)
        assert not any(value.is_meta or value.device.type!="cuda" for value in self.model.parameters())
        assert not any(value.is_meta for value in self.model.buffers())
        for layer in self.model.layers:
            if hasattr(layer,"linear_attn"):
                layer.linear_attn.chunk_gated_delta_rule=delta
                # The causal_conv1d extension is absent. The native fallback
                # retains identical causal convolution/SiLU operations.
                layer.linear_attn.causal_conv1d_fn=None
        assert all(not p.requires_grad for layer in self.model.layers[:56] for p in layer.parameters())
        self.norms=nn.ModuleList([nn.LayerNorm(5120) for _ in self.depths]).to("cuda")
        self.plain=nn.Linear(40960,18).to("cuda");self.centered=nn.Linear(40960,18).to("cuda")
        self.tokenizer=AutoTokenizer.from_pretrained(base,local_files_only=True)
        self.tokenizer.pad_token_id=248044;self.tokenizer.padding_side="right"
        self.load_report=dict(revision=REVISION,seconds=time.perf_counter()-began,packed_linears=packed_count,
            ignored_original_BF16_linears=dense_count,packed_gpu_bytes=packed_gpu_bytes,
            CPU_packed_embedding_bytes=self.model.embed_tokens.cpu_packed_bytes,
            trainable_parameters=sum(p.numel() for p in self.parameters() if p.requires_grad),
            delta_backend=delta.__module__,causal_convolution="native PyTorch",pad_multiple=pad_multiple,
            all_decoder_layers=64,LoRA_target_layers=list(range(56,64)),
            dtype="FP16 compute/FP32 GSQ scales+LoRA+norms+heads; BF16 decoded on CPU",
            stale_unused_MTP_index_entries=len(self.reader.stale_mtp_keys))

    def forward(self,token_rows):
        assert all(ids and ids[-1]==248044 for ids in token_rows)
        batch=self.tokenizer.pad({"input_ids":token_rows},padding=True,pad_to_multiple_of=self.pad_multiple,return_tensors="pt")
        assert batch.attention_mask.sum(1).tolist()==[len(ids) for ids in token_rows]
        assert batch.input_ids.shape[1]<=self.cfg.max_position_embeddings
        raw=[]
        with torch.no_grad():
            hidden=self.model.embed_tokens(batch.input_ids)
            attention=batch.attention_mask.to("cuda")
            pos=torch.arange(hidden.shape[1],device="cuda")[None].expand(hidden.shape[0],-1)
            kwargs=dict(config=self.cfg,inputs_embeds=hidden,attention_mask=attention,past_key_values=None,position_ids=pos)
            masks={"full_attention":qm.create_causal_mask(**kwargs),"linear_attention":qm.create_recurrent_attention_mask(**kwargs)}
            rope=self.model.rotary_emb(hidden,pos[None].expand(3,-1,-1))
            mask=attention.unsqueeze(-1).float();denominator=mask.sum(1)
            for index in range(56):
                with torch.autocast("cuda",dtype=torch.float16):
                    hidden=self.model.layers[index](hidden,position_embeddings=rope,
                        attention_mask=masks[self.cfg.layer_types[index]],position_ids=pos,past_key_values=None,use_cache=False)
                if index+1 in self.depths:raw.append((hidden.float()*mask).sum(1)/denominator)
        assert len(raw)==7 and not hidden.requires_grad
        for index in range(56,64):
            layer=self.model.layers[index]
            def apply(value,block=layer,rotary=rope,position=pos,attention_mask=masks[self.cfg.layer_types[index]]):
                with torch.autocast("cuda",dtype=torch.float16):
                    return block(value,position_embeddings=rotary,attention_mask=attention_mask,
                                 position_ids=position,past_key_values=None,use_cache=False)
            hidden=apply(hidden)
        raw.append((hidden.float()*mask).sum(1)/denominator)
        features=torch.cat([norm(value) for norm,value in zip(self.norms,raw)],-1)
        assert features.shape==(len(token_rows),40960) and torch.isfinite(features).all()
        return features

In [ ]:
%%writefile /kaggle/working/gsq/gsq_infer_layers.py
"""Authentic GSQ packed weights with bounded FP16 T4 inference.

The3bit Triton decoding is the same operator that passed the authentic k_proj
BF16 forward/dX/LoRA proof. No re-quantization or full-model decompression.
"""
import math
import torch
import triton
import triton.language as tl
from torch import nn
from torch.nn import functional as F


@triton.jit
def unpack3_kernel(P,S,W,N:tl.constexpr,K:tl.constexpr,PACKED_COLS:tl.constexpr,GROUPS:tl.constexpr,BLOCK:tl.constexpr):
    element=tl.program_id(0)*BLOCK+tl.arange(0,BLOCK)
    valid=element<N*K
    row=element//K;col=element%K
    bit=col*3;word=bit//32;shift=bit%32
    low=tl.load(P+row*PACKED_COLS+word,mask=valid,other=0).to(tl.uint32)
    crosses=shift>29
    high=tl.load(P+row*PACKED_COLS+word+1,mask=valid&crosses,other=0).to(tl.uint32)
    unsigned=((low>>shift)|(high<<((32-shift)&31)))&7
    signed=unsigned.to(tl.int32)-4
    scale=tl.load(S+row*GROUPS+col//128,mask=valid,other=0).to(tl.float32)
    tl.store(W+element,signed.to(tl.float32)*scale,mask=valid)


def dequantize(packed,scales):
    n,groups=scales.shape;k=groups*128
    output=torch.empty((n,k),device=packed.device,dtype=torch.float16)
    unpack3_kernel[(triton.cdiv(n*k,512),)](packed,scales,output,n,k,packed.shape[1],groups,512)
    return output


class GSQLinear(nn.Module):
    def __init__(self,packed,scales,in_features,out_features,rank):
        super().__init__()
        assert packed.dtype==torch.int32 and scales.dtype==torch.float32
        assert tuple(packed.shape)==(out_features,in_features*3//32)
        assert tuple(scales.shape)==(out_features,in_features//128)
        self.register_buffer("packed",packed,persistent=False)
        self.register_buffer("scales",scales,persistent=False)
        self.in_features,self.out_features=in_features,out_features
        self.bias=None
        initialize_lora(self,rank)
    def forward(self,value):
        base=F.linear(value,dequantize(self.packed,self.scales))
        return add_lora(self,value,base)


class DenseFrozenLinear(nn.Module):
    """Original ignored BF16 in_proj_a/b weights; not re-quantized."""
    def __init__(self,weight,in_features,out_features,rank):
        super().__init__()
        assert tuple(weight.shape)==(out_features,in_features)
        self.register_buffer("base_weight",weight,persistent=False)
        self.in_features,self.out_features=in_features,out_features
        self.bias=None
        initialize_lora(self,rank)
    def forward(self,value):
        return add_lora(self,value,F.linear(value,self.base_weight))


def initialize_lora(module,rank):
    module.scale=2.
    module.dropout=nn.Identity()
    if rank:
        module.lora_A=nn.Parameter(torch.empty(rank,module.in_features,device="cuda",dtype=torch.float32))
        module.lora_B=nn.Parameter(torch.zeros(module.out_features,rank,device="cuda",dtype=torch.float32))
        nn.init.kaiming_uniform_(module.lora_A,a=math.sqrt(5))
    else:
        module.register_parameter("lora_A",None);module.register_parameter("lora_B",None)


def add_lora(module,value,base):
    if module.lora_A is None:return base
    with torch.autocast("cuda",dtype=torch.float16):
        extra=F.linear(F.linear(module.dropout(value),module.lora_A),module.lora_B)
    return base+module.scale*extra.to(base.dtype)

In [ ]:
%%writefile /kaggle/working/gsq/delta_turing.py
# Copyright 2025 The Qwen Team and The HuggingFace Inc. team. All rights reserved.
# Modifications copyright 2026 Vladislav Lavrentev.
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy at http://www.apache.org/licenses/LICENSE-2.0
# Unless required by applicable law or agreed to in writing, software distributed
# under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR
# CONDITIONS OF ANY KIND, either express or implied. See the License for the
# specific language governing permissions and limitations under the License.
"""Qwen/Hugging Face DeltaNet reference with batched triangular solve on T4.

Only the strict-lower inverse recurrence is changed. The remainder of the
reference chunk algorithm is retained verbatim, including all chunk states.
"""
import torch
from torch.nn import functional as F
from transformers.models.qwen3_5.modeling_qwen3_5 import l2norm

def triangular_chunk_gated_delta_rule(
    query,
    key,
    value,
    g,
    beta,
    chunk_size=64,
    initial_state=None,
    output_final_state=False,
    use_qk_l2norm_in_kernel=False,
    **kwargs,
):
    initial_dtype = query.dtype
    if use_qk_l2norm_in_kernel:
        query = l2norm(query, dim=-1, eps=1e-6)
        key = l2norm(key, dim=-1, eps=1e-6)
    query, key, value, beta, g = [
        x.transpose(1, 2).contiguous().to(torch.float32) for x in (query, key, value, beta, g)
    ]

    batch_size, num_heads, sequence_length, k_head_dim = key.shape
    v_head_dim = value.shape[-1]
    pad_size = (chunk_size - sequence_length % chunk_size) % chunk_size
    query = F.pad(query, (0, 0, 0, pad_size))
    key = F.pad(key, (0, 0, 0, pad_size))
    value = F.pad(value, (0, 0, 0, pad_size))
    beta = F.pad(beta, (0, pad_size))
    g = F.pad(g, (0, pad_size))
    total_sequence_length = sequence_length + pad_size
    scale = 1 / (query.shape[-1] ** 0.5)
    query = query * scale

    v_beta = value * beta.unsqueeze(-1)
    k_beta = key * beta.unsqueeze(-1)
    # reshape to chunks
    query, key, value, k_beta, v_beta = [
        x.reshape(x.shape[0], x.shape[1], -1, chunk_size, x.shape[-1]) for x in (query, key, value, k_beta, v_beta)
    ]
    g = g.reshape(g.shape[0], g.shape[1], -1, chunk_size)
    mask = torch.triu(torch.ones(chunk_size, chunk_size, dtype=torch.bool, device=query.device), diagonal=0)

    # chunk decay
    g = g.cumsum(dim=-1)
    decay_mask = ((g.unsqueeze(-1) - g.unsqueeze(-2)).tril().exp().float()).tril()
    attn = -((k_beta @ key.transpose(-1, -2)) * decay_mask).masked_fill(mask, 0)
    # If A is the original strictly-lower triangular attn, the reference
    # recurrence computes B=A+A*B, hence I+B=(I-A)^(-1). A batched triangular
    # solve replaces only those63 Python row updates. Keep this solve FP32
    # even inside the T4 FP16 autocast context; no FP16 triangular solver.
    with torch.autocast(device_type=query.device.type, enabled=False):
        identity = torch.eye(chunk_size, dtype=torch.float32, device=attn.device)
        coefficient = identity - attn.float()
        identity = identity.expand_as(coefficient)
        attn = torch.linalg.solve_triangular(coefficient, identity,
                                             upper=False, unitriangular=True)
    value = attn @ v_beta
    k_cumdecay = attn @ (k_beta * g.exp().unsqueeze(-1))
    last_recurrent_state = (
        torch.zeros(batch_size, num_heads, k_head_dim, v_head_dim, dtype=value.dtype, device=value.device)
        if initial_state is None
        else initial_state.to(value)
    )
    core_attn_out = torch.zeros_like(value)
    mask = torch.triu(torch.ones(chunk_size, chunk_size, dtype=torch.bool, device=query.device), diagonal=1)

    # for each chunk
    for i in range(0, total_sequence_length // chunk_size):
        q_i, k_i, v_i = query[:, :, i], key[:, :, i], value[:, :, i]
        attn = q_i @ k_i.transpose(-1, -2) * decay_mask[:, :, i]
        v_prime = (k_cumdecay[:, :, i]) @ last_recurrent_state
        v_new = v_i - v_prime
        attn_inter = (q_i * g[:, :, i, :, None].exp()) @ last_recurrent_state
        core_attn_out[:, :, i] = attn_inter + attn @ v_new
        last_recurrent_state = (
            last_recurrent_state * g[:, :, i, -1, None, None].exp()
            + (k_i * (g[:, :, i, -1, None] - g[:, :, i]).exp()[..., None]).transpose(-1, -2) @ v_new
        )

    if not output_final_state:
        last_recurrent_state = None
    core_attn_out = core_attn_out.reshape(core_attn_out.shape[0], core_attn_out.shape[1], -1, core_attn_out.shape[-1])
    core_attn_out = core_attn_out[:, :, :sequence_length]
    core_attn_out = core_attn_out.transpose(1, 2).contiguous().to(initial_dtype)
    return core_attn_out, last_recurrent_state


In [ ]:
%%writefile /kaggle/working/gsq/adapter.py
"""Exact final GSQ adapter contract; no partial checkpoint fallback."""
import json
from pathlib import Path

import torch
from safetensors import safe_open

REVISION = "b5ce0b76f60020a875dee4f6ec9d934cca4121e4"


def load_final(model, final_file, metadata_file, complete_file):
    final_file, metadata_file, complete_file = map(Path, (final_file, metadata_file, complete_file))
    for path in (final_file, metadata_file, complete_file):
        if not path.is_file():
            raise FileNotFoundError(f"Required final training artifact missing: {path}")
    metadata = json.loads(metadata_file.read_text(encoding="utf-8"))
    complete = json.loads(complete_file.read_text(encoding="utf-8"))
    assert complete["status"] == "TRAINED_FULLTEXT_GSQ3_PUBLIC119899_OFFICIAL2_EPOCHS"
    assert complete["rows_seen"] == {"external": 119899, "official": 5398}
    assert complete["final_artifact"]["path"] == final_file.name == "final.safetensors"
    assert complete["contract_hash"] == metadata["checkpoint_contract_hash"]
    assert metadata["source_revision"] == REVISION
    assert metadata["rank"] == 16 and metadata["alpha"] == 32
    assert metadata["depths"] == [8, 16, 24, 32, 40, 48, 56, 64]
    assert metadata["lora_target_layer_indices"] == list(range(56, 64))
    assert metadata["executed_decoder_layers"] == 64 and metadata["text_truncation"] is False
    assert len(metadata["labels"]) == 18
    assert final_file.stat().st_size == complete["final_artifact"]["bytes"]
    expected = {name: param for name, param in model.named_parameters() if param.requires_grad}
    export_keys = {"backbone." + name.removeprefix("model.") if name.startswith("model.") else name: name
                   for name in expected}
    with safe_open(str(final_file), framework="pt", device="cpu") as weights:
        assert set(weights.keys()) == set(export_keys), "Final adapter tensor names differ from model trainable parameters"
        with torch.no_grad():
            for key, name in export_keys.items():
                source = weights.get_tensor(key)
                target = expected[name]
                assert source.dtype == target.dtype and source.shape == target.shape and torch.isfinite(source).all(), key
                target.copy_(source.to(target.device))
    model.eval()
    return metadata

In [ ]:
%%writefile /kaggle/working/gsq/infer.py
"""Two isolated T4 workers for full-text GSQ features and exact row reassembly."""
import argparse
import json
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd


def read_rows(path, text_column):
    frame = pd.read_parquet(path).reset_index(drop=True)
    assert text_column in frame and frame[text_column].notna().all()
    texts = frame[text_column].astype(str).tolist()
    assert all(texts)
    return frame, texts


def worker(args):
    assert os.environ.get("CUDA_VISIBLE_DEVICES") in ("0", "1")
    import torch
    from adapter import load_final
    from gsq_infer_model import GSQClassifier

    assert torch.cuda.device_count() == 1, "GSQ worker must see one isolated GPU"
    assert torch.cuda.get_device_capability(0) >= (7, 5), "GSQ T4 path requires SM 7.5 or newer"
    torch.set_grad_enabled(False)
    frame, texts = read_rows(args.input, args.text_column)
    indices = np.arange(len(texts), dtype=np.int64)[args.worker_id::2]
    assert len(indices)
    model = GSQClassifier(args.base)
    metadata = load_final(model, args.final, args.metadata, args.complete)
    tokenizer = model.tokenizer
    features = np.empty((len(indices), 40960), dtype=np.float32)
    plain = np.empty((len(indices), 18), dtype=np.float32)
    for begin in range(0, len(indices), args.batch_size):
        batch_ids = indices[begin:begin + args.batch_size]
        token_rows = tokenizer([texts[int(i)] for i in batch_ids], add_special_tokens=False,
                               truncation=False)["input_ids"]
        token_rows = [ids + [248044] for ids in token_rows]
        assert all(len(ids) <= model.cfg.max_position_embeddings for ids in token_rows), "Full text exceeds model context"
        with torch.inference_mode():
            value = model(token_rows)
            score = model.plain(value)
        stop = begin + len(batch_ids)
        features[begin:stop] = value.float().cpu().numpy()
        plain[begin:stop] = score.float().cpu().numpy()
        del value, score
    args.output.parent.mkdir(parents=True, exist_ok=True)
    np.savez(args.output, row_index=indices, features=features, plain_logits=plain)
    print(json.dumps(dict(worker=args.worker_id, rows=len(indices), labels=metadata["labels"],
                          load_report=model.load_report)), flush=True)


def group_codes(args, frame):
    assert (args.groups is None) != (args.prompt_column is None), "Provide exactly one group source"
    if args.groups is not None:
        codes = np.load(args.groups, allow_pickle=False)
        assert codes.ndim == 1 and len(codes) == len(frame) and np.issubdtype(codes.dtype, np.integer)
        assert np.all(codes >= 0), "Explicit group codes must be nonnegative"
        return codes
    assert args.prompt_column in frame, "Requested prompt column is missing"
    assert frame[args.prompt_column].notna().all(), "Missing prompt group is ambiguous"
    return pd.factorize(frame[args.prompt_column], sort=False)[0]


def combine(args, outputs):
    from safetensors import safe_open

    frame, _ = read_rows(args.input, args.text_column)
    n = len(frame)
    features = np.empty((n, 40960), dtype=np.float32)
    plain = np.empty((n, 18), dtype=np.float32)
    seen = np.zeros(n, dtype=bool)
    for path in outputs:
        with np.load(path, allow_pickle=False) as item:
            ids = item["row_index"]
            assert ids.ndim == 1 and np.issubdtype(ids.dtype, np.integer)
            assert len(np.unique(ids)) == len(ids)
            assert np.all((0 <= ids) & (ids < n)) and not np.any(seen[ids])
            assert item["features"].shape == (len(ids), 40960)
            assert item["plain_logits"].shape == (len(ids), 18)
            features[ids] = item["features"]
            plain[ids] = item["plain_logits"]
            seen[ids] = True
    assert seen.all() and np.isfinite(features).all() and np.isfinite(plain).all()
    metadata = json.loads(args.metadata.read_text(encoding="utf-8"))
    with safe_open(str(args.final), framework="pt", device="cpu") as weights:
        kernel = weights.get_tensor("centered.weight").numpy().astype(np.float32)
        bias = weights.get_tensor("centered.bias").numpy().astype(np.float32)
    assert kernel.shape == (18, 40960) and bias.shape == (18,)
    groups = group_codes(args, frame)
    centered = plain.copy()
    valid_centered = np.zeros(n, dtype=bool)
    for group in np.unique(groups):
        ids = np.flatnonzero(groups == group)
        if len(ids) == 1:
            continue  # No contrast: downstream uses plain logits and mask=False.
        centered[ids] = (features[ids] - features[ids].mean(0, keepdims=True)) @ kernel.T + bias
        valid_centered[ids] = True
    assert np.isfinite(centered).all() and len(groups) == n
    args.output.parent.mkdir(parents=True, exist_ok=True)
    np.savez(args.output, row_index=np.arange(n, dtype=np.int64),
             plain_logits=plain, centered_logits=centered, valid_centered=valid_centered,
             group_code=groups, labels=np.asarray(metadata["labels"]))


def parent(args):
    _, texts = read_rows(args.input, args.text_column)
    worker_count = min(2, len(texts))
    outputs = [args.work_dir / f"gsq-worker-{i}.npz" for i in range(worker_count)]
    args.work_dir.mkdir(parents=True, exist_ok=True)
    children = []
    for device in range(worker_count):
        command = [sys.executable, str(Path(__file__).resolve()), "--worker-id", str(device),
                   "--base", str(args.base), "--final", str(args.final), "--metadata", str(args.metadata),
                   "--complete", str(args.complete), "--input", str(args.input), "--output", str(outputs[device]),
                   "--text-column", args.text_column,
                   "--batch-size", str(args.batch_size)]
        env = os.environ.copy()
        env.update(CUDA_VISIBLE_DEVICES=str(device), HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1",
                   TOKENIZERS_PARALLELISM="false")
        children.append(subprocess.Popen(command, env=env))
    codes = [child.wait() for child in children]
    if any(code != 0 for code in codes):
        raise RuntimeError(f"GSQ workers failed: exit codes {codes}; outputs were not combined")
    combine(args, outputs)


def main():
    p = argparse.ArgumentParser(description=__doc__)
    for name in ("base", "final", "metadata", "complete", "input", "output"):
        p.add_argument("--" + name, type=Path, required=True)
    p.add_argument("--work-dir", type=Path)
    p.add_argument("--text-column", default="text")
    p.add_argument("--prompt-column")
    p.add_argument("--groups", type=Path)
    p.add_argument("--batch-size", type=int, default=1)
    p.add_argument("--worker-id", type=int, choices=(0, 1))
    args = p.parse_args()
    assert args.batch_size >= 1
    if args.worker_id is None:
        assert args.work_dir is not None and os.environ.get("CUDA_VISIBLE_DEVICES") in (None, "", "0,1")
        assert (args.groups is None) != (args.prompt_column is None), "Provide --groups or --prompt-column"
        parent(args)
    else:
        worker(args)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile infer.py
"""End-to-end attribution pipeline (self-contained; runs on Kaggle T4x2 or locally in --sim mode).
1. Mean-pooled hidden states from frozen LMs for train+test (computed here so precision/quantisation match).
2. Per model: LR on raw + LR on prompt-centred features (train centred by prompt_id, test by estimated cluster).
3. TF-IDF (char_wb + word/punct) prompt-centred LR.
4. Test prompts recovered by size-capped average-linkage clustering on content-word TF-IDF.
5. Sum log-probs, then Hungarian assignment inside each cluster (each model answers each prompt at most once).
"""
import os, sys, time, gc, argparse, json, re
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import numpy as np, pandas as pd, scipy.sparse as sp, torch
from scipy.optimize import linear_sum_assignment

LABELS = ['claude-fable-5','claude-opus-4-6','claude-opus-4-8','claude-opus-5','claude-sonnet-5',
 'gemini-3.1-pro-preview','gemini-3.6-flash','gemini-3.7-flash','gemini-3.8-flash','glm-5.3','glm-5.3-flash',
 'gpt-5.5','gpt-5.6-luna','gpt-5.6-sol','gpt-5.6-terra','gpt-6-astra','kimi-k3','qwen3.8-max']
L2I = {l: i for i, l in enumerate(LABELS)}

p = argparse.ArgumentParser()
p.add_argument('--data', default='/kaggle/input/competitions/name-that-llm')
p.add_argument('--models', default='')   # json list of {"path":..,"layers":[fractions],"quant":"fp16|4bit|8bit","bs":16}
p.add_argument('--comp_path', default=''); p.add_argument('--comp_w', type=float, default=0.0)
p.add_argument('--helper_path', default=''); p.add_argument('--helper_w', type=float, default=0.0)
p.add_argument('--gsq_path', default=''); p.add_argument('--gsq_lp_w', type=float, default=0.0); p.add_argument('--gsq_prob_w', type=float, default=0.0)
p.add_argument('--member_deadline', type=float, default=0.0)
p.add_argument('--fake_test', type=int, default=0, help='timing probe: score N pool passages as a stand-in test')
p.add_argument('--atlas_probe', type=int, default=0, help='timing probe: stop ATLAS after N passages')
p.add_argument('--sim', type=str, default='', help='comma list of folds held out of train.csv as fake test')
p.add_argument('--out', default='submission.parquet')
p.add_argument('--cache', default='cache')
p.add_argument('--dtype', default='fp16')
p.add_argument('--C', type=float, default=0.005)
p.add_argument('--tfidf_w', type=float, default=1.0)
p.add_argument('--distort_w', type=float, default=0.0, help='weight of topic-masked (text distortion) TF-IDF member')
p.add_argument('--adapt_thr', type=float, default=0.15, help='cluster cut (avg cosine sim on content-word TF-IDF)')
p.add_argument('--adapt_w', type=float, default=0.0, help='weight of centred twin evidence on rows in clusters of size>=2 (0=off)')
p.add_argument('--post', type=int, default=0, help='1: spaced em-dash guard + Hungarian on dense test clusters (12-18)')
p.add_argument('--atlas_dir', default=''); p.add_argument('--atlas_base', default=''); p.add_argument('--atlas_w', default='0', help='mix weight per bundle (comma list or one value)')
p.add_argument('--atlas_margin', type=float, default=0.0, help='>0: score ATLAS only on rows whose top-2 margin is below this')
p.add_argument('--atlas_bs', type=int, default=4); p.add_argument('--atlas_chunk', type=int, default=3); p.add_argument('--atlas_budget', type=float, default=8.3 * 3600)
p.add_argument('--sk_div', type=int, default=1, help='1: divide blend by #members before Sinkhorn (ours); 0: raw blend (public notebook)')
p.add_argument('--sinkhorn', type=float, default=0.0, help='mix strength for damped marginal balancing (0=off)')
p.add_argument('--sinkhorn_target_mix', type=float, default=0.35, help='shrink predicted class marginal toward official-train prior; 0=prediction marginal, 1=train prior')
p.add_argument('--sim_k', default='', help="sim only: keep 'lo-hi' random passages per held-out prompt")
p.add_argument('--extra', default='', help='parquet/csv (text, model) of matched external passages, used by TF-IDF/distort members')
p.add_argument('--extra_w', type=float, default=0.5)
p.add_argument('--svc_w', type=float, default=0.0, help='4-view n-gram LinearSVC on train + --pool')
p.add_argument('--pool', default='')
p.add_argument('--save_lp', default='', help='also save normalised blend log-probs (n x 18, LABELS order) to this .npy')
p.add_argument('--perpassage', type=int, default=1, help='1: test passages treated independently (no clustering/centring/assignment)')
a, _ = p.parse_known_args()
os.makedirs(a.cache, exist_ok=True)
torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False   # fp32 sums inside fp16 matmuls (free on T4)
T0 = time.time()
def log(*x): print(f'[{time.time()-T0:7.0f}s]', *x, flush=True)

# ---------------- data
train = pd.read_csv(os.path.join(a.data, 'train.csv'))
if a.sim:
    rng = np.random.RandomState(0); pids = np.array(sorted(train.prompt_id.unique())); rng.shuffle(pids)
    pf = {q: i % 5 for i, q in enumerate(pids)}; fo = train.prompt_id.map(pf).values
    hold = np.isin(fo, [int(x) for x in a.sim.split(',')])
    if a.sim_k:
        lo_, hi_ = map(int, a.sim_k.split('-')); keep = np.zeros(len(train), bool); idx_h = np.where(hold)[0]
        for g in np.unique(train.prompt_id.values[idx_h]):
            ii = idx_h[train.prompt_id.values[idx_h] == g]; keep[rng.choice(ii, min(len(ii), rng.randint(lo_, hi_ + 1)), replace=False)] = True
        hold = keep | (hold & False)
    test = train[hold].drop(columns=['model']).reset_index(drop=True); test_truth = train[hold].reset_index(drop=True)
    hold_any = np.isin(fo, [int(x) for x in a.sim.split(',')])
    train = train[~hold_any].reset_index(drop=True)
elif a.fake_test:
    test = pd.read_parquet(a.pool).sample(a.fake_test, random_state=0)[['text']].reset_index(drop=True)
    test['id'] = [f'fake{i}' for i in range(len(test))]
else:
    test = pd.read_csv(os.path.join(a.data, 'test.csv'))
train['text'] = train.text.fillna(''); test['text'] = test.text.fillna('')
y = train.model.map(L2I).values; Gtr = train.prompt_id.values
log('train', len(train), 'test', len(test))

# ---------------- utils
def center_groups(X, groups):
    X = X.copy()
    for g in np.unique(groups):
        i = groups == g; X[i] -= X[i].mean(0)
    return X

def torch_lr(Xtr, ytr, Xte, C, iters=200):
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    Xtr = torch.as_tensor(Xtr, dtype=torch.float32, device=dev); Xte = torch.as_tensor(Xte, dtype=torch.float32, device=dev)
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-6; Xtr = (Xtr - mu) / sd; Xte = (Xte - mu) / sd
    yt = torch.as_tensor(ytr, dtype=torch.long, device=dev)
    W = torch.zeros(Xtr.shape[1], 18, device=dev, requires_grad=True); b = torch.zeros(18, device=dev, requires_grad=True)
    opt = torch.optim.LBFGS([W, b], lr=1, max_iter=iters, history_size=20, line_search_fn='strong_wolfe')
    def closure():
        opt.zero_grad()
        loss = torch.nn.functional.cross_entropy(Xtr @ W + b, yt, reduction='sum') + 0.5 / C * (W ** 2).sum()
        loss.backward(); return loss
    opt.step(closure)
    with torch.no_grad(): return torch.log_softmax(Xte @ W + b, -1).cpu().numpy()

def capped_agglo(S, cap=18, thr=0.02):
    """Average linkage (Lance-Williams), never merging past `cap` members; stop when best avg sim < thr."""
    n = len(S); A = S.astype(np.float32).copy(); np.fill_diagonal(A, -np.inf)
    size = np.ones(n, np.int32); alive = np.ones(n, bool); lab = np.arange(n)
    while True:
        ok = (size[:, None] + size[None, :] <= cap) & alive[:, None] & alive[None, :]
        M = np.where(ok, A, -np.inf); k = int(np.argmax(M)); i, j = divmod(k, n)
        if not np.isfinite(M[i, j]) or M[i, j] < thr: break
        new = (size[i] * A[i] + size[j] * A[j]) / (size[i] + size[j])
        A[i] = new; A[:, i] = new; A[i, i] = -np.inf; A[j] = -np.inf; A[:, j] = -np.inf
        size[i] += size[j]; alive[j] = False; lab[lab == j] = i
    return lab

def hungarian(lp, groups):
    pred = lp.argmax(1).copy()
    for g in np.unique(groups):
        idx = np.where(groups == g)[0]; r, c = linear_sum_assignment(-lp[idx]); pred[idx[r]] = c
    return pred

# ---------------- 1. recover test prompts
from sklearn.feature_extraction.text import TfidfVectorizer
if a.perpassage and a.adapt_w <= 0:
    Gte = np.arange(len(test))
else:
  try:
    v = TfidfVectorizer(sublinear_tf=True, stop_words='english', min_df=2, max_df=0.05)
    Xc = v.fit_transform(test.text.values)
    Gte = capped_agglo((Xc @ Xc.T).toarray(), 18, a.adapt_thr if a.perpassage else 0.02)
  except ValueError as e:  # tiny interactive test file: nothing survives df pruning
    log('clustering fallback:', e); Gte = np.arange(len(test))
log('test clusters', len(np.unique(Gte)), 'sizes', np.bincount(np.bincount(Gte)[np.bincount(Gte) > 0]).tolist())
if a.sim:
    from sklearn.metrics import adjusted_rand_score
    log('cluster ARI vs truth', round(adjusted_rand_score(test_truth.prompt_id, Gte), 4))

LP = {}
EXT_T, EXT_Y = np.array([], dtype=object), np.array([], dtype=int)
EXTRA_DF = pd.DataFrame(columns=['text', 'model'])
def _norm_text_key(value):
    import unicodedata
    return re.sub(r'\s+', ' ', unicodedata.normalize('NFKC', str(value or ''))).strip()
if a.extra:
    _E = pd.read_parquet(a.extra) if a.extra.endswith('.parquet') else pd.read_csv(a.extra)
    if not {'text', 'model'}.issubset(_E.columns):
        raise ValueError(f'External file must contain text/model columns: {a.extra}')
    _E = _E[_E.model.isin(LABELS)].copy()
    _E['text'] = _E.text.fillna('').astype(str)
    # In --sim mode, remove any external exact-text matches to held-out official rows.
    # For final submission, retain the public external pool as provided by the source.
    if a.sim:
        _held_keys = {_norm_text_key(t) for t in test.text.values}
        _E = _E[~_E.text.map(_norm_text_key).isin(_held_keys)].copy()
    _official_keys = {_norm_text_key(t) for t in train.text.values}
    _seen = set(_official_keys)
    _keep = []
    for _t in _E.text.values:
        _k = _norm_text_key(_t)
        _ok = bool(_k) and _k not in _seen
        _keep.append(_ok)
        if _ok: _seen.add(_k)
    EXTRA_DF = _E.loc[_keep, ['text', 'model']].reset_index(drop=True)
    EXT_T = EXTRA_DF.text.astype(str).values
    EXT_Y = EXTRA_DF.model.map(L2I).values.astype(int)
    log('matched-clean extra rows after exact de-dup', len(EXT_T))
FIT_T = np.concatenate([train.text.values, EXT_T])
FIT_Y = np.concatenate([y, EXT_Y])
FIT_W = np.concatenate([np.ones(len(train), dtype=np.float32), np.full(len(EXT_T), a.extra_w, dtype=np.float32)])
# ---------------- 2. TF-IDF centred LR
if a.tfidf_w > 0:
    from sklearn.linear_model import LogisticRegression
    mats_tr, mats_te = [], []
    for kw in [dict(analyzer='char_wb', ngram_range=(2, 5), min_df=3, max_features=60000),
               dict(token_pattern=r"\w+|[^\w\s]", ngram_range=(1, 2), min_df=3, max_features=30000)]:
        vec = TfidfVectorizer(sublinear_tf=True, lowercase=False, **kw).fit(FIT_T if a.perpassage else train.text.values)
        mats_tr.append(vec.transform(FIT_T if a.perpassage else train.text.values)); mats_te.append(vec.transform(test.text.values))
    if a.perpassage:
        LP['tfidf'] = LogisticRegression(C=20, max_iter=3000).fit(sp.hstack(mats_tr).tocsr(), FIT_Y, sample_weight=FIT_W).predict_log_proba(sp.hstack(mats_te).tocsr()) * a.tfidf_w
    else:
        Xtr = center_groups(sp.hstack(mats_tr).toarray().astype(np.float32), Gtr)
        Xte = center_groups(sp.hstack(mats_te).toarray().astype(np.float32), Gte)
        LP['tfidf'] = LogisticRegression(C=5, max_iter=3000).fit(Xtr, y).predict_log_proba(Xte) * a.tfidf_w
        del Xtr, Xte
    gc.collect(); log('tfidf done')
else:
    LP['tfidf'] = np.zeros((len(test), 18), dtype=np.float32)
    log('tfidf skipped (zero ensemble weight)')
if a.svc_w > 0:  # four-view SVC: official + matched-clean + pool, with source weights and exact de-duplication
    import re as _re
    from scipy.sparse import hstack as _hs
    from sklearn.svm import LinearSVC
    _FW = set("a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little".split())
    _shape = lambda t: _re.sub(r"[^\W\d_]+", lambda m_: "X" if m_[0].isupper() else "x", t)
    _func = lambda t: " ".join(w.lower() if w.lower() in _FW or not w.isalnum() else "CONTENT" for w in _re.findall(r"\w+|[^\w\s]", t))
    _parts = []
    _official = train[['text', 'model']].copy(); _official['text'] = _official.text.fillna('').astype(str)
    _official['_srcw'] = 1.0
    _parts.append(_official)
    _seen = {_norm_text_key(t) for t in _official.text.values}
    if len(EXTRA_DF):
        _e = EXTRA_DF[['text', 'model']].copy(); _e['_srcw'] = float(a.extra_w)
        _keep = []
        for _t in _e.text.values:
            _k = _norm_text_key(_t); _ok = bool(_k) and _k not in _seen
            _keep.append(_ok)
            if _ok: _seen.add(_k)
        _e = _e.loc[_keep].copy(); _parts.append(_e)
    if a.pool and os.path.exists(a.pool):
        _P = pd.read_parquet(a.pool)
        if not {'text', 'model'}.issubset(_P.columns):
            raise ValueError(f'Pool must contain text/model columns: {a.pool}')
        _P = _P[_P.model.isin(LABELS)][['text', 'model']].copy()
        _P['text'] = _P.text.fillna('').astype(str); _P['_srcw'] = 0.35
        if a.sim:
            _held_keys = {_norm_text_key(t) for t in test.text.values}
            _P = _P[~_P.text.map(_norm_text_key).isin(_held_keys)].copy()
        _keep = []
        for _t in _P.text.values:
            _k = _norm_text_key(_t); _ok = bool(_k) and _k not in _seen
            _keep.append(_ok)
            if _ok: _seen.add(_k)
        _P = _P.loc[_keep].copy(); _parts.append(_P)
    _fitdf = pd.concat(_parts, ignore_index=True)
    _fit = _fitdf.text.astype(str).tolist(); _fy = _fitdf.model.map(L2I).to_numpy(dtype=int); _sw = _fitdf['_srcw'].to_numpy(dtype=np.float32)
    log(f'svc4 fit rows={len(_fit)} source_weighted; external clean={len(EXTRA_DF)}')
    _A, _B = [], []
    for view, lo, hi, mf, wv in [("char", 3, 5, 250000, 1.0), ("word", 1, 3, 180000, 0.7), ("shape", 2, 6, 60000, 0.5), ("function", 1, 3, 50000, 0.5)]:
        prep = _shape if view == "shape" else _func if view == "function" else (lambda s_: s_)
        an = "char" if view == "shape" else "word" if view == "function" else view
        o = dict(analyzer=an, ngram_range=(lo, hi), lowercase=False, sublinear_tf=True, min_df=2, max_features=mf, dtype=np.float32)
        if an == "word": o["token_pattern"] = r"(?u)\b\w+\b|[^\w\s]"
        _v = TfidfVectorizer(**o); _A.append(_v.fit_transform([prep(t) for t in _fit]) * wv); _B.append(_v.transform([prep(t) for t in test.text.values]) * wv)
    _svc = LinearSVC(C=0.5, random_state=42, max_iter=5000, class_weight='balanced')
    _svc.fit(_hs(_A, format='csr'), _fy, sample_weight=_sw)
    _d = _svc.decision_function(_hs(_B, format='csr'))
    LP['svc4'] = (_d - _d.max(1, keepdims=True) - np.log(np.exp(_d - _d.max(1, keepdims=True)).sum(1, keepdims=True))) * a.svc_w
    del _A, _B, _parts, _fitdf, _svc; gc.collect(); log('svc4 done')
if a.distort_w > 0:  # Stamatatos-style text distortion: keep top-100 frequent words + punctuation, mask the rest
    import re, collections
    wre = re.compile(r"[A-Za-z’']+")
    cnt = collections.Counter(w.lower() for t in FIT_T for w in wre.findall(t))
    keep = {w for w, _ in cnt.most_common(100)}
    dist = lambda arr: [wre.sub(lambda m_: m_.group(0) if m_.group(0).lower() in keep else '*', t) for t in arr]
    Dtr, Dte = dist(FIT_T), dist(test.text.values)
    vs = [TfidfVectorizer(sublinear_tf=True, lowercase=False, analyzer='char', ngram_range=(2, 6), min_df=2),
          TfidfVectorizer(sublinear_tf=True, lowercase=True, token_pattern=r"[^\s]+", ngram_range=(1, 3), min_df=2)]
    Xa = sp.hstack([v_.fit_transform(Dtr) for v_ in vs]).tocsr(); Xb = sp.hstack([v_.transform(Dte) for v_ in vs]).tocsr()
    LP['distort'] = LogisticRegression(C=10, max_iter=3000).fit(Xa, FIT_Y, sample_weight=FIT_W).predict_log_proba(Xb) * a.distort_w
    log('distort done')

# ---------------- 3. LM embeddings
from transformers import AutoTokenizer, AutoModel
@torch.no_grad()
def embed(spec, texts):
    tok = AutoTokenizer.from_pretrained(spec['path'])
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    kw = dict(device_map='auto')
    q = spec.get('quant', 'fp16')
    if q == '4bit':
        from transformers import BitsAndBytesConfig
        kw['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
    elif q == '8bit':
        from transformers import BitsAndBytesConfig
        kw['quantization_config'] = BitsAndBytesConfig(load_in_8bit=True)
    kw['dtype'] = torch.bfloat16 if q == 'bf16' else torch.float16
    m = AutoModel.from_pretrained(spec['path'], **kw).eval()
    enc = tok(list(texts), truncation=True, max_length=512)['input_ids']
    order = np.argsort([len(x) for x in enc]); out = None; bs = spec.get('bs', 16); nan_rows = 0
    first = next(m.parameters()).device
    for i in range(0, len(order), bs):
        b = order[i:i + bs]; L = max(len(enc[j]) for j in b)
        ids = torch.full((len(b), L), tok.pad_token_id); att = torch.zeros((len(b), L), dtype=torch.long)
        for r, j in enumerate(b): ids[r, :len(enc[j])] = torch.tensor(enc[j]); att[r, :len(enc[j])] = 1
        hs = m(input_ids=ids.to(first), attention_mask=att.to(first), output_hidden_states=True).hidden_states
        nL = len(hs) - 1; sel = spec['layers_abs'] if 'layers_abs' in spec else [max(1, round(f * nL)) for f in spec['layers']]
        pooled = []
        for l in sel:
            h = hs[l].float(); w = att.to(h.device).unsqueeze(-1).float()
            pooled.append(((h * w).sum(1) / w.sum(1)).cpu())
        pooled = torch.cat(pooled, 1).numpy()
        nan_rows += int((~np.isfinite(pooled)).any(1).sum())
        if out is None: out = np.zeros((len(texts), pooled.shape[1]), np.float32); log('  layers', sel, 'of', nL)
        out[b] = pooled
    del m; gc.collect(); torch.cuda.empty_cache()
    if nan_rows: log('  WARNING non-finite rows:', nan_rows)
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)

TPL = 'Identify which AI language model wrote the following passage, based only on its writing style.\n\nPassage:\n{}\n\nThe author model is:'
@torch.no_grad()
def lora_predict(spec, texts):
    """LoRA classifier WITHOUT peft: merge W += (B @ A) * alpha/r into the base weights, load the trained score head."""
    from transformers import AutoModelForSequenceClassification
    from safetensors.torch import load_file
    tok = AutoTokenizer.from_pretrained(spec['path'])
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    m = AutoModelForSequenceClassification.from_pretrained(spec['path'], num_labels=18, dtype=torch.float16, device_map='auto')
    m.config.pad_token_id = tok.pad_token_id
    cfg = json.load(open(os.path.join(spec['adapter'], 'adapter_config.json')))
    scale = cfg['lora_alpha'] / cfg['r']
    sd = load_file(os.path.join(spec['adapter'], 'adapter_model.safetensors'))
    mods = dict(m.named_modules()); n_merged = 0; head = None
    for k, v in sd.items():
        key = k.replace('base_model.model.', '', 1)
        if '.lora_A.' in key:
            mod_name = key.split('.lora_A.')[0]; B = sd[k.replace('.lora_A.', '.lora_B.')]
            lin = mods[mod_name]; W = lin.weight
            W += ((B.to(W.device, torch.float32) @ v.to(W.device, torch.float32)) * scale).to(W.dtype); n_merged += 1
        elif key.startswith('score.') and 'weight' in key and 'original_module' not in key:
            head = v
    assert head is not None and n_merged > 0, (n_merged, [k for k in sd if 'score' in k])
    m.score.weight.copy_(head.to(m.score.weight.device, m.score.weight.dtype))
    log(f'  merged {n_merged} LoRA pairs (scale {scale}), head {tuple(head.shape)}')
    del mods, sd, lin, W, B, head; gc.collect()  # drop every reference to the model's modules
    m.eval()
    enc = tok(list(texts), truncation=True, max_length=512)['input_ids']
    order = np.argsort([len(x) for x in enc]); out = np.zeros((len(texts), 18), np.float32); bs = spec.get('bs', 16)
    first = next(m.parameters()).device
    for i in range(0, len(order), bs):
        b = order[i:i + bs]; L = max(len(enc[j]) for j in b)
        ids = torch.full((len(b), L), tok.pad_token_id); att = torch.zeros((len(b), L), dtype=torch.long)
        for r, j in enumerate(b): ids[r, :len(enc[j])] = torch.tensor(enc[j]); att[r, :len(enc[j])] = 1
        lo = m(input_ids=ids.to(first), attention_mask=att.to(first)).logits.float()
        out[b] = torch.log_softmax(lo, -1).cpu().numpy()
    del m; gc.collect(); torch.cuda.empty_cache()
    return out

@torch.no_grad()
def l24_predict(spec, texts, groups=None, train_texts=None):
    """LoRA24-recipe member(s): truncated base loaded ONCE; for each adapter dir in spec['adapters'] (or spec['adapter'])
    merge W += B A * s, compute masked mean-pooled multi-layer features, read plain (+ centred) heads, then unmerge.
    Returns log-probs averaged over adapters (plain, centred-or-None)."""
    from transformers import AutoConfig
    from safetensors.torch import load_file
    from concurrent.futures import ThreadPoolExecutor
    adapters = spec.get('adapters') or [spec['adapter']]
    tok = AutoTokenizer.from_pretrained(spec['path'])
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    cfg = AutoConfig.from_pretrained(spec['path']); tc = cfg.text_config if hasattr(cfg, 'text_config') else cfg
    tc.num_hidden_layers = spec['keep']; tc.use_cache = False
    if getattr(tc, 'layer_types', None) is not None: tc.layer_types = tc.layer_types[:spec['keep']]
    _ms = []   # fork (exp018): with spec['dp'], a full copy on each GPU if the largest batch fits with 1 GiB to spare
    if spec.get('dp') and torch.cuda.device_count() >= 2:
        try:
            for _d in range(2):
                _ms.append(AutoModel.from_pretrained(spec['path'], config=cfg, dtype=torch.float16, device_map={'': _d}).eval())
                for _n in ('vision_tower', 'multi_modal_projector'):   # Ministral 3's image path; text passes never use it
                    if getattr(_ms[-1], _n, None) is not None: setattr(_ms[-1], _n, None)
                gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(_d)
                _ms[-1](input_ids=torch.full((spec.get('bs', 16), spec.get('max_len', 384)), tok.pad_token_id, device=f'cuda:{_d}'), output_hidden_states=True, use_cache=False)
                assert torch.cuda.max_memory_allocated(_d) < torch.cuda.get_device_properties(_d).total_memory - (1 << 30), 'under 1 GiB to spare'
        except Exception as _e:   # out of memory, or anything else: upstream's way
            log(f'  l24 {spec["name"]}: one split copy instead of one per GPU ({_e!r:.80})'); _ms = []
        gc.collect(); torch.cuda.empty_cache()
    if not _ms: _ms = [AutoModel.from_pretrained(spec['path'], config=cfg, dtype=torch.float16, device_map='auto').eval()]
    m = _ms[0]
    if spec.get('no_final_norm'):
        for _m in _ms: _m.norm = torch.nn.Identity()
    mods = dict(m.named_modules()); first = next(m.parameters()).device; L = spec['layers']
    _mods = [dict(_m.named_modules()) for _m in _ms]
    _eos = spec.get('eos_id')   # checkpoints trained with an end token appended (Nishant's) need the same id at inference
    _tk = lambda tx: [x + ([_eos] if _eos is not None else []) for x in tok(list(tx), truncation=True, max_length=spec.get('max_len', 384) - (1 if _eos is not None else 0))['input_ids']]
    enc = _tk(texts)
    order = np.argsort([len(x) for x in enc]); bs = spec.get('bs', 16)
    def lsm(z): return z - z.max(1, keepdims=True) - np.log(np.exp(z - z.max(1, keepdims=True)).sum(1, keepdims=True))
    P_acc, C_acc = 0.0, 0.0; FEATS = None; OVF = [0, 0]
    if train_texts is not None:
        enc_tr = _tk(train_texts)
    def featurize(enc_, norms, hid):
        order_ = np.argsort([len(x) for x in enc_]); Z_ = np.zeros((len(enc_), hid * len(L)), np.float32)
        def _run(k, n):   # fork: upstream's batches in upstream's order, batch i on copy i % n, one thread per copy
            mm = _ms[k]; dev = next(mm.parameters()).device
            with torch.no_grad():   # grad mode is per thread
                for i in range(k * bs, len(order_), bs * n):
                    b = order_[i:i + bs]; Lm = max(len(enc_[j]) for j in b)
                    ids = torch.full((len(b), Lm), tok.pad_token_id); att = torch.zeros((len(b), Lm), dtype=torch.long)
                    for r, j in enumerate(b): ids[r, :len(enc_[j])] = torch.tensor(enc_[j]); att[r, :len(enc_[j])] = 1
                    hs = mm(input_ids=ids.to(dev), attention_mask=att.to(dev), output_hidden_states=True, use_cache=False).hidden_states
                    if not all(torch.isfinite(hs[l]).all() for l in L):   # fp16 overflow (un-normed deep states): redo this batch in bf16
                        mm.to(torch.bfloat16); hs = mm(input_ids=ids.to(dev), attention_mask=att.to(dev), output_hidden_states=True, use_cache=False).hidden_states; mm.to(torch.float16); OVF[k] += 1
                    mask = att.unsqueeze(-1).float(); mask[:, 0] = 0; _mg = {d_: mask.to(d_) for d_ in {hs[l].device for l in L}}
                    Z_[b] = torch.cat([nm(((hs[l].float() * _mg[hs[l].device]).sum(1) / _mg[hs[l].device].sum(1)).cpu()) for l, nm in zip(L, norms)], -1).numpy()   # fork: masked means on the GPU
        if len(_ms) == 1: _run(0, 1)
        else:
            try:
                with ThreadPoolExecutor(len(_ms)) as _ex:
                    for _f in [_ex.submit(_run, k, len(_ms)) for k in range(len(_ms))]: _f.result()
            except Exception as _e:   # never worse than one GPU: drop the second copy, every batch again on the first
                log(f'  l24 {spec["name"]}: two-copy pass failed ({_e!r:.80}), one copy from here on'); del _ms[1:], _mods[1:]
                gc.collect(); torch.cuda.empty_cache(); _run(0, 1)
        return Z_
    n_used = 0
    for ai, ad in enumerate(adapters):
        if ai > 0 and a.member_deadline > 0 and time.time() - T0 > a.member_deadline:
            log(f'  l24 {spec["name"]}: time guard triggered, {ai}/{len(adapters)} adapters used'); break
        n_used += 1
        adp = os.path.join(ad, 'adapter') if os.path.exists(os.path.join(ad, 'adapter', 'adapter_config.json')) else ad
        acfg = json.load(open(os.path.join(adp, 'adapter_config.json'))); scale = acfg['lora_alpha'] / acfg['r']
        sd = load_file(os.path.join(adp, 'adapter_model.safetensors')); deltas = []
        for k, v in sd.items():
            if '.lora_A.' not in k: continue
            name = k.replace('base_model.model.', '', 1).split('.lora_A.')[0]
            if name not in mods and ('model.' + name) in mods: name = 'model.' + name
            B = sd[k.replace('.lora_A.', '.lora_B.')]
            for _md in _mods:   # fork: every copy
                W = _md[name].weight
                W += ((B.to(W.device, torch.float32) @ v.to(W.device, torch.float32)) * scale).to(W.dtype)
                deltas.append((W, B, v))   # keep only the low-rank factors; the dense delta is recomputed to unmerge
        if os.path.exists(os.path.join(ad, 'heads.pt')): H = torch.load(os.path.join(ad, 'heads.pt'), map_location='cpu')
        else: H = dict(load_file(os.path.join(ad, 'heads.safetensors')))
        if os.path.exists(os.path.join(ad, 'meta.json')):   # remap head rows from the checkpoint's label order to ours
            _lab = json.load(open(os.path.join(ad, 'meta.json')))['labels']; _perm = [_lab.index(l) for l in LABELS]
            for _p in ('plain', 'centered'): H[f'{_p}.weight'] = H[f'{_p}.weight'][_perm]; H[f'{_p}.bias'] = H[f'{_p}.bias'][_perm]
        hid = H['norms.0.weight'].shape[0]
        norms = [torch.nn.LayerNorm(hid) for _ in L]
        for i, nm in enumerate(norms): nm.load_state_dict({'weight': H[f'norms.{i}.weight'], 'bias': H[f'norms.{i}.bias']})
        Z = featurize(enc, norms, hid)   # fork: upstream's loop, now shared with the Gaussian head's train features
        if train_texts is not None and ai == len(adapters) - 1:   # full-data adapter: features for the Gaussian head
            FEATS = (Z.copy(), featurize(enc_tr, norms, hid))
        for W, B, A_ in deltas:   # unmerge before the next adapter
            W -= ((B.to(W.device, torch.float32) @ A_.to(W.device, torch.float32)) * scale).to(W.dtype)
        torch.cuda.empty_cache()
        del deltas, sd; gc.collect()
        hd = lambda p, X: lsm(X @ H[f'{p}.weight'].float().numpy().T + H[f'{p}.bias'].float().numpy())
        P_acc = P_acc + hd('plain', Z) / len(adapters)
        if groups is not None:
            Zc = Z.copy()
            for g in np.unique(groups):
                ii = groups == g; Zc[ii] -= Zc[ii].mean(0)
            C_acc = C_acc + hd('centered', Zc) / len(adapters)
        log(f'  l24 {spec["name"]}: adapter {os.path.basename(ad)} done (bf16 redo batches {sum(OVF)})')
    del m, mods, _ms, _mods; gc.collect(); torch.cuda.empty_cache()
    if 0 < n_used < len(adapters):
        P_acc = P_acc * (len(adapters) / n_used)
        if groups is not None:
            C_acc = C_acc * (len(adapters) / n_used)
    if train_texts is not None: return P_acc, (C_acc if groups is not None else None), FEATS
    return P_acc, (C_acc if groups is not None else None)

def gpu_mem():
    return ' '.join(f'gpu{i}:{torch.cuda.memory_allocated(i)/1e9:.1f}GB' for i in range(torch.cuda.device_count()))

specs = json.loads(a.models) if a.models else []
all_text = np.concatenate([train.text.values, test.text.values])
for spec in specs:
    if spec.get('type') == 'l24':
        log('l24', spec['name'])
        if len(test) < 50:
            _p40, _ = l24_predict(spec, train.text.values[:40]); _acc = float((_p40.argmax(1) == y[:40]).mean())
            log(f'  {spec["name"]} in-sample check acc {_acc:.2f}'); assert _acc > 0.5, f'{spec["name"]} l24 does not reproduce its fit'
        if spec.get('gauss', 0) > 0:
            plain, cent, _feats_g = l24_predict(spec, test.text.values, Gte if a.adapt_w > 0 else None, train_texts=train.text.values)
            if _feats_g is not None:
                Zte_g, Ztr_g = _feats_g
                import gauss as _ga
                prm = _ga.fit(Ztr_g, y, Gtr, dict(red='wpca', d=128, se_shrink='lw', sp_shrink='lw', lam_floor=1e-4, n_exact=3, inj=True, iters=5, repel=True, sp_scale=1.0))
                prm['T1'] = prm['Tg'] = 3.0
                _sz = np.bincount(Gte)[Gte]; _rows = (_sz >= 2) & (_sz <= 18)
                _q = _ga.scores(Ztr_g[[0, 1]], np.zeros(2, dtype=int), prm); assert np.allclose(np.exp(_q).sum(1), 1)
                _G = np.zeros((len(test), 18))
                if _rows.any():
                    _G[_rows] = _ga.scores(Zte_g[_rows], np.unique(Gte[_rows], return_inverse=True)[1], prm)
                LP[spec['name'] + '_gauss'] = _G * spec['gauss']; log(f"  gaussian head {spec['name']}: {int(_rows.sum())} grouped test rows, weight {spec['gauss']}")
        else:
            plain, cent = l24_predict(spec, test.text.values, Gte if a.adapt_w > 0 else None)
        LP[spec['name']] = plain * spec.get('w', 1.0)
        if cent is not None:
            rows = np.bincount(Gte)[Gte] >= 2
            LP[spec['name'] + '_twin'] = np.where(rows[:, None], cent, 0.0) * a.adapt_w * spec.get('w', 1.0) * spec.get('twin', 1.0)
        gc.collect(); torch.cuda.empty_cache(); log(spec['name'], 'done', gpu_mem()); continue
    if spec.get('type') == 'enc':   # fully fine-tuned encoder (DeBERTa-style): fp32, mean of the last layer, linear head
        log('enc', spec['name']); from safetensors.torch import load_file as _lf
        _ck = spec['ckpt']; _meta = json.load(open(os.path.join(_ck, 'meta.json')))
        _perm = [_meta['labels'].index(l) for l in LABELS]
        _hd = _lf(os.path.join(_ck, 'head.safetensors')); _W = _hd['weight'].float().numpy()[_perm]; _b = _hd['bias'].float().numpy()[_perm]
        _etok = AutoTokenizer.from_pretrained(os.path.join(_ck, 'encoder'))
        _enc = AutoModel.from_pretrained(os.path.join(_ck, 'encoder'), dtype=torch.float32).to('cuda:0').eval()
        def _enc_feats(texts):
            _ids = _etok(list(texts), truncation=True, max_length=_meta['max_len'])['input_ids']
            _ord = np.argsort([len(x) for x in _ids], kind='stable'); Z_ = np.zeros((len(_ids), _enc.config.hidden_size), np.float32)
            with torch.inference_mode():
                for _s in range(0, len(_ord), 32):
                    _idx = _ord[_s:_s + 32]; _bt = _etok.pad({'input_ids': [_ids[i] for i in _idx]}, return_tensors='pt').to('cuda:0')
                    _m = _bt['attention_mask'].unsqueeze(-1).float()
                    Z_[_idx] = ((_enc(**_bt).last_hidden_state * _m).sum(1) / _m.sum(1).clamp(min=1)).cpu().numpy()
            return Z_
        if len(test) < 50:
            _acc = float(((_enc_feats(train.text.values[:40]) @ _W.T + _b).argmax(1) == y[:40]).mean())
            log(f'  {spec["name"]} in-sample check acc {_acc:.2f}'); assert _acc > 0.5, spec['name']
        _zz = _enc_feats(test.text.values) @ _W.T + _b
        LP[spec['name']] = (_zz - _zz.max(1, keepdims=True) - np.log(np.exp(_zz - _zz.max(1, keepdims=True)).sum(1, keepdims=True))) * spec.get('w', 1.0)
        del _enc; gc.collect(); torch.cuda.empty_cache(); log(spec['name'], 'done', gpu_mem()); continue
    if spec.get('type') == 'head':
        log('head', spec['name']); H = np.load(spec['head'])
        Ete = embed(dict(spec, layers=sorted(float(v) for v in H['layers'])), test.text.values)  # extraction stored layers in sorted order
        if len(test) < 50:  # interactive run: in-sample sanity check on 40 train rows (head was fit on them)
            _E40 = embed(dict(spec, layers=sorted(float(v) for v in H['layers'])), train.text.values[:40])
            _acc = float(((((_E40 - H['mu']) / H['sd']) @ H['W'] + H['b']).argmax(1) == y[:40]).mean())
            log(f'  {spec["name"]} in-sample check acc {_acc:.2f}'); assert _acc > 0.5, f'{spec["name"]} head does not reproduce its fit'
        Z = (Ete - H['mu']) / H['sd']; zz = Z @ H['W'] + H['b']
        LP[spec['name']] = (zz - zz.max(1, keepdims=True) - np.log(np.exp(zz - zz.max(1, keepdims=True)).sum(1, keepdims=True))) * spec.get('w', 1.0)
        gc.collect(); torch.cuda.empty_cache(); log(spec['name'], 'done', gpu_mem()); continue
    if spec.get('type') == 'lora':
        log('lora', spec['name']); LP[spec['name']] = lora_predict(spec, test.text.values) * spec.get('w', 1.0)
        gc.collect(); torch.cuda.empty_cache(); log(spec['name'], 'done', gpu_mem()); continue
    name = spec['name']; cp = os.path.join(a.cache, f"{name}_{'sim' + a.sim + a.sim_k if a.sim else 'real'}.npy")
    if os.path.exists(cp): E = np.load(cp)
    else:
        log('embedding', name); E = embed(spec, all_text); np.save(cp, E)
    Etr, Ete = E[:len(train)], E[len(train):]
    r = torch_lr(Etr, y, Ete, spec.get('C', a.C))
    if a.perpassage:
        LP[name] = r * spec.get('w', 1.0)
        if a.adapt_w > 0:
            sizes = np.bincount(Gte)[Gte]; rows = sizes >= 2
            c = torch_lr(center_groups(Etr, Gtr), y, center_groups(Ete, Gte), spec.get('C', a.C))
            LP[name + '_twin'] = np.where(rows[:, None], c, 0.0) * a.adapt_w * spec.get('w', 1.0)
    else:
        c = torch_lr(center_groups(Etr, Gtr), y, center_groups(Ete, Gte), a.C)
        LP[name] = (r + c) * spec.get('w', 1.0)
    gc.collect(); torch.cuda.empty_cache(); log(name, 'done', gpu_mem())

# ---------------- 4. combine + assign
# typography guards: (mark, models that ever use it in train). Zero violations on the 38k matched pool for every rule.
GUARDS = [(' \u2014 ', {'claude-opus-4-6', 'claude-opus-5', 'glm-5.3', 'glm-5.3-flash'}),
          ('\u2019', {l for l in LABELS if not l.startswith(('claude', 'glm'))}),
          ('"', {'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'glm-5.3', 'glm-5.3-flash', 'gpt-5.5', 'kimi-k3'}),
          ('\u201c', {'gemini-3.8-flash', 'gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'kimi-k3', 'qwen3.8-max'}),
          ('\u2013', {'claude-opus-5', 'gemini-3.1-pro-preview', 'glm-5.3-flash', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'kimi-k3'})]
GSQ_PROB = None
if a.gsq_path and os.path.exists(a.gsq_path):
    try:
        with np.load(a.gsq_path, allow_pickle=False) as _gz:
            assert np.array_equal(_gz['row_index'], np.arange(len(test)))
            _glab = _gz['labels'].astype(str).tolist()
            _gperm = [_glab.index(l) for l in LABELS]
            _gp = _gz['plain_logits'][:, _gperm].astype(np.float64)
            _gc = _gz['centered_logits'][:, _gperm].astype(np.float64)
            _gp_lp = _gp - _gp.max(1, keepdims=True) - np.log(np.exp(_gp - _gp.max(1, keepdims=True)).sum(1, keepdims=True))
            _gc_lp = _gc - _gc.max(1, keepdims=True) - np.log(np.exp(_gc - _gc.max(1, keepdims=True)).sum(1, keepdims=True))
            if a.gsq_lp_w > 0:
                LP['gsq27b'] = _gp_lp * a.gsq_lp_w
                if a.adapt_w > 0:
                    _grows = np.bincount(Gte)[Gte] >= 2
                    LP['gsq27b_twin'] = np.where(_grows[:, None], _gc_lp, 0.0) * a.adapt_w * a.gsq_lp_w * 0.35
                log(f'gsq27b added to core LP with w={a.gsq_lp_w}')
            _gp_prob = np.exp(_gp_lp); _gp_prob /= _gp_prob.sum(1, keepdims=True)
            _gc_prob = np.exp(_gc_lp); _gc_prob /= _gc_prob.sum(1, keepdims=True)
            _grows = np.bincount(Gte)[Gte] >= 2
            GSQ_PROB = np.where(_grows[:, None], 0.78 * _gp_prob + 0.22 * _gc_prob, _gp_prob)
            GSQ_PROB /= GSQ_PROB.sum(1, keepdims=True)
    except Exception as _ge:
        log('GSQ SKIPPED:', repr(_ge)); GSQ_PROB = None
tot = sum(LP.values())
def atlas_scores(bundle, rows, budget):
    """Source-conditioned likelihood probabilities (public ATLAS bundle format) for test rows `rows`, in our label order.
    Rows are scored shortest-first; completed rows are returned even when budget prevents finishing all rows."""
    sys.path.insert(0, bundle)
    for _m in [m_ for m_ in list(sys.modules) if m_.startswith(('ntl_atlas', 'ntl_rlcd'))]: del sys.modules[_m]
    from ntl_atlas.model import load_model as _atlas_load
    from ntl_rlcd.data import LABELS as _alab
    from safetensors.torch import load_file as _lf
    sys.path.remove(bundle)
    _cfg = json.load(open(os.path.join(bundle, 'atlas_bundle.json')))['training_config']
    _am, _atok = _atlas_load(a.atlas_base, _alab, _cfg, device='auto')
    try:
        _am.load_trainable(_lf(os.path.join(bundle, 'adapters.safetensors'))); _am.eval()
        _enc = _atok(test.text.values[rows].tolist(), truncation=True, max_length=_cfg['max_length'])['input_ids']
        _ord = np.argsort([len(t) for t in _enc], kind='stable')
        out = np.zeros((len(rows), 18), np.float32)
        scored = np.zeros(len(rows), dtype=bool)
        _t1 = time.time()
        _ctok = np.cumsum(np.array([len(_enc[int(i)]) for i in _ord], np.float64))
        for _b in range(0, len(_ord), a.atlas_bs):
            if _b and time.time() - T0 >= budget - 60:
                log(f'atlas budget checkpoint: preserving {int(scored.sum())}/{len(rows)} completed predictions')
                break
            _idx = _ord[_b:_b + a.atlas_bs]
            _pk = _atok.pad({'input_ids': [_enc[int(i)] for i in _idx]}, padding=True, return_tensors='pt').to(_am.device)
            with torch.inference_mode(): _p, _ = _am.score(_pk['input_ids'], _pk['attention_mask'], source_chunk=a.atlas_chunk)
            _batch = _p.float().cpu().numpy()
            if not np.isfinite(_batch).all() or not np.allclose(_batch.sum(1), 1, atol=1e-3):
                log(f'atlas invalid scores in batch {_b}; keeping previously completed rows')
                break
            out[_idx] = _batch
            scored[_idx] = True
            _done = _b + len(_idx)
            if _b % (25 * a.atlas_bs) == 0 and _done > 0:
                _proj = (time.time() - T0) + (time.time() - _t1) / max(float(_ctok[_done - 1]), 1.0) * float(_ctok[-1] - _ctok[_done - 1]) * 1.1
                log(f'atlas {os.path.basename(bundle)} {_done}/{len(_ord)} projected finish {_proj:.0f}s')
                if a.atlas_probe and _done >= a.atlas_probe:
                    log(f'atlas probe stopped after {int(scored.sum())} scored rows')
                    break
        assert not scored.any() or np.isfinite(out[scored]).all()
        log(f'atlas completed {int(scored.sum())}/{len(rows)} rows before budget')
        return out[:, [_alab.index(l) for l in LABELS]], scored
    finally:
        del _am; gc.collect(); torch.cuda.empty_cache()

COMP_P = None
if a.comp_w > 0 and a.comp_path and os.path.exists(a.comp_path):   # complementary public stack, run in its own process
    try:
        _cm = json.load(open(a.comp_path.replace('comp_logit.npy', 'comp_meta.json')))
        assert _cm['ids'] == test['id'].astype(str).tolist(), 'comp stack row order differs'
        _cl = np.load(a.comp_path)[:, [_cm['labels'].index(l) for l in LABELS]]
        COMP_P = np.exp(_cl - _cl.max(1, keepdims=True)); COMP_P /= COMP_P.sum(1, keepdims=True)
        assert np.isfinite(COMP_P).all(); log(f"comp stack loaded ({_cm['seconds']:.0f}s)")
    except Exception as _e:
        log('COMP SKIPPED:', repr(_e)); COMP_P = None
elif a.comp_w > 0: log('COMP SKIPPED: no stack output')
_bundles = [d for d in a.atlas_dir.split(',') if d]; _aw = [float(x) for x in str(a.atlas_w).split(',')] if _bundles else []
if _bundles and len(_aw) == 1: _aw = _aw * len(_bundles)
if (a.sinkhorn > 0 and len(test) >= 18) or _bundles or COMP_P is not None:
    wsum = sum(abs(v).max() > 0 for v in LP.values())
    lpn = tot / max(1, wsum) if a.sk_div else tot; P = np.exp(lpn - lpn.max(1, keepdims=True)); P /= P.sum(1, keepdims=True)
    if COMP_P is not None: P = (1 - a.comp_w) * P + a.comp_w * COMP_P; P /= P.sum(1, keepdims=True)
    if _bundles and sum(_aw) > 0:
        _s = np.sort(P, 1); _margin = _s[:, -1] - _s[:, -2]
        rows = np.where(_margin < a.atlas_margin)[0] if a.atlas_margin > 0 and len(test) >= 50 else np.arange(len(test))
        log(f'atlas on {len(rows)}/{len(test)} rows (top-2 margin < {a.atlas_margin})')
        mixed = P.copy(); _wleft = 1.0
        for _bd, _w in zip(_bundles, _aw):
            try:
                A_, scored = atlas_scores(_bd, rows, a.atlas_budget)
                completed_rows = rows[scored]
                if len(completed_rows):
                    mixed[completed_rows] += _w * (A_[scored] - P[completed_rows])
                log(f'atlas {_bd} mixed {len(completed_rows)}/{len(rows)} rows at {_w}', gpu_mem())
            except Exception as _e:
                log('ATLAS SKIPPED:', _bd, repr(_e))
        P = mixed / mixed.sum(1, keepdims=True)
    if GSQ_PROB is not None and a.gsq_prob_w > 0:
        P = (1 - a.gsq_prob_w) * P + a.gsq_prob_w * GSQ_PROB; P /= P.sum(1, keepdims=True)
        log(f'gsq27b probability mixed at {a.gsq_prob_w}')
    if a.helper_w > 0 and a.helper_path and os.path.exists(a.helper_path):
        try:
            with np.load(a.helper_path) as hz:
                _pos = {i: r for r, i in enumerate(hz['ids'].astype(str))}; _hl = hz['labels'].astype(str).tolist()
                H_ = hz['probabilities'][[_pos[i] for i in test['id'].astype(str)]][:, [_hl.index(l) for l in LABELS]]
            assert H_.shape == P.shape and np.isfinite(H_).all() and np.allclose(H_.sum(1), 1, atol=1e-4)
            P = (1 - a.helper_w) * P + a.helper_w * H_; P /= P.sum(1, keepdims=True)
            log(f'spectral readout heads mixed at {a.helper_w}')
        except Exception as _e:
            log('spectral readout heads skipped:', repr(_e))
    Q = P.copy()
    if not (a.sinkhorn > 0 and len(test) >= 18): tot = np.log(P + 1e-12)
    if a.sinkhorn > 0 and len(test) >= 18:
        # The competition states test class counts are unequal and Public/Private mixes differ.
        # Avoid forcing a uniform marginal: shrink model-implied test marginals toward train priors.
        _rho = float(np.clip(a.sinkhorn_target_mix, 0.0, 1.0))
        _pred_marg = np.clip(P.mean(0), 1e-6, None); _pred_marg /= _pred_marg.sum()
        _train_prior = np.bincount(y, minlength=18).astype(np.float64) + 1.0
        _train_prior /= _train_prior.sum()
        _target = (1.0 - _rho) * _pred_marg + _rho * _train_prior
        _target = np.clip(_target, 1e-6, None); _target /= _target.sum()
        for _ in range(100):
            Q *= (len(Q) * _target[None, :]) / np.clip(Q.sum(0, keepdims=True), 1e-12, None)
            Q /= np.clip(Q.sum(1, keepdims=True), 1e-12, None)
        tot = (1 - a.sinkhorn) * np.log(P + 1e-12) + a.sinkhorn * np.log(Q + 1e-12)
_v38_before_hard = tot.copy()
if a.post and a.perpassage:
    _nfix = 0
    for _mark, _allowed in GUARDS:
        _forb = np.array([l not in _allowed for l in LABELS])
        for _i, _t in enumerate(test.text.values):
            if _mark in _t and LABELS[int(tot[_i].argmax())] not in _allowed:
                tot[_i, _forb] = -1e9; _nfix += 1
    _sz = np.bincount(Gte)[Gte]; _nh = 0
    for _g in np.unique(Gte[(_sz >= 12) & (_sz <= 18)]):
        _ii = np.where(Gte == _g)[0]; _r, _c = linear_sum_assignment(-tot[_ii]); _nh += int((tot[_ii].argmax(1)[_r] != _c).sum())
        tot[_ii[_r]] = -1e9; tot[_ii[_r], _c] = 0.0
    log(f'post: em-dash guard fixed {_nfix}, dense-cluster Hungarian changed {_nh}')
pred = tot.argmax(1) if a.perpassage else hungarian(tot, Gte)
if a.sim:
    from sklearn.metrics import balanced_accuracy_score as bas
    yt = test_truth.model.map(L2I).values
    for k, v_ in LP.items(): log(f'  {k:12s} arg {bas(yt, v_.argmax(1)):.4f} hung {bas(yt, hungarian(v_, Gte)):.4f}')
    log(f'ENSEMBLE arg {bas(yt, tot.argmax(1)):.4f} | hung(est clusters) {bas(yt, pred):.4f} | hung(true prompts) {bas(yt, hungarian(tot, test_truth.prompt_id.values)):.4f}')
    np.save(os.path.join(a.cache, f'sim{a.sim}_lp.npy'), tot)
if a.save_lp:
    wtot = sum(abs(float(s_.get('w', 1.0))) for s_ in specs) + (a.tfidf_w if a.tfidf_w > 0 else 0) + (a.distort_w if a.distort_w > 0 else 0) + (a.svc_w if a.svc_w > 0 else 0)
    from scipy.special import log_softmax as _lsm
    np.save(a.save_lp, _lsm(tot / max(wtot, 1e-6), axis=1).astype(np.float32)); log('saved', a.save_lp, 'norm', wtot)
sub = pd.DataFrame({'id': test.id.values, 'model': [LABELS[i] for i in pred]})
assert sub.model.isin(LABELS).all() and sub.id.is_unique and len(sub) == len(test)
sub.to_parquet(a.out, index=False); log('wrote', a.out, sub.model.value_counts().to_dict())

if not a.sim:
    import pathlib as _V38Path
    import scipy.special as _v38sp
    import json as _v38json
    _out = _V38Path.Path(a.out)
    _marks = [' — ', '—', '–', '’', '‘', '“', '”', '«', '»', '"', ';', ':', '(', ')', '[', ']', '**', '`', '…', '•', '\n', '\n\n', 'e.g.', 'i.e.', 'However,', 'Overall,', 'In conclusion', 'Therefore,', 'Moreover,', '\t', ' - ', ' = ', '\n- ', '\n1.', '###', '```', '> ']
    _Xtr = np.array([[float(_m in _t) for _m in _marks] for _t in train.text.values], dtype=np.float32)
    _Xte = np.array([[float(_m in _t) for _m in _marks] for _t in test.text.values], dtype=np.float32)
    _freq = np.bincount(y, minlength=len(LABELS)).astype(np.float64)
    _hits = np.stack([_Xtr[y == _k].sum(axis=0) for _k in range(len(LABELS))], axis=0).astype(np.float64)
    # Hierarchical Beta-Bernoulli smoothing: class-specific marker rates shrink toward the global rate.
    _global_rate = (_Xtr.sum(axis=0) + 2.0) / (len(_Xtr) + 4.0)
    _kappa = 24.0
    _rate = (_hits + _kappa * _global_rate[None, :]) / (_freq[:, None] + _kappa)
    _rate = np.clip(_rate, 0.01, 0.99)
    _global_rate = np.clip(_global_rate, 0.01, 0.99)
    _log_rate = np.log(_rate); _log_not_rate = np.log1p(-_rate)
    _base_log_rate = np.log(_global_rate); _base_log_not_rate = np.log1p(-_global_rate)
    # Weight markers by how consistently they separate classes, capped to avoid one marker dominating.
    _signal = np.sqrt(np.mean((_rate - _global_rate[None, :]) ** 2, axis=0))
    _fw = _signal / max(float(np.median(_signal[_signal > 0])) if np.any(_signal > 0) else 1.0, 1e-6)
    _fw = np.clip(_fw, 0.25, 2.0)
    # Matrix form: weighted Bernoulli log likelihood ratio for each class.
    _evidence = (_Xte * _fw[None, :]) @ _log_rate.T + ((1.0 - _Xte) * _fw[None, :]) @ _log_not_rate.T
    _evidence -= (((_Xte * _fw[None, :]) @ _base_log_rate) + (((1.0 - _Xte) * _fw[None, :]) @ _base_log_not_rate))[:, None]
    _evidence -= _evidence.mean(axis=1, keepdims=True)
    _evidence = np.clip(1.5 * _evidence / max(float(_fw.sum()), 1e-6), -0.30, 0.30)
    _raw = _v38_before_hard.copy()
    _probs = _v38sp.softmax(_raw, axis=1)
    _sorted = np.sort(_probs, axis=1)
    _margin = _sorted[:, -1] - _sorted[:, -2]
    _confidence_gate = np.clip((0.60 - _margin) / 0.60, 0.0, 1.0)
    _word_count = np.array([len(re.findall(r'\b\w+\b', str(_t))) for _t in test.text.values], dtype=np.float32)
    _length_gate = np.clip(np.sqrt(np.maximum(_word_count, 1.0) / 80.0), 0.40, 1.0)
    _active_markers = _Xte.sum(axis=1)
    _coverage_gate = np.clip(np.sqrt((_active_markers + 1.0) / 4.0), 0.45, 1.0)
    _gate = _confidence_gate * _length_gate * _coverage_gate
    _safe_logits = _raw + _gate[:, None] * _evidence
    _reference_logits = _safe_logits.copy()
    try:
        from sibling_style import apply_grouped_style_residual
        _safe_logits = apply_grouped_style_residual(
            _safe_logits, train.text.values, test.text.values, LABELS, y, Gtr,
            max_alpha=0.06, diagnostics_path=str(_out.with_name('v40a_style_oof.json')))
        log('V40A family-only OOF residual fitted')
    except Exception as _style_error:
        log('V40A family-only residual failed; using exact inherited prediction:', repr(_style_error))
        _safe_logits = _reference_logits
    def _v31_sub(z):
        return pd.DataFrame({'id': test.id.values, 'model': [LABELS[int(i)] for i in z.argmax(axis=1)]})
    _raw_sub = _v31_sub(_raw)
    _safe = _v31_sub(_safe_logits)
    _safe.to_parquet(_out, index=False)
    assert _safe.id.is_unique and len(_safe) == len(test) and _safe.model.isin(LABELS).all()
    with open(_out.with_name('v31_diagnostics.json'), 'w') as _fo:
        _v38json.dump({
            'method': 'hierarchical_beta_bernoulli_typography_v2',
            'fit_scope': 'full official train labels, used only for final submission inference; not an OOF validation claim',
            'pre_bayes_changed': int((_raw_sub.model.values != _safe.model.values).sum()),
            'trained_rows': int(len(train)), 'tested_rows': int(len(test)),
            'marker_count': int(len(_marks)), 'mean_gate': float(_gate.mean()),
            'length_gate_mean': float(_length_gate.mean()), 'coverage_gate_mean': float(_coverage_gate.mean()),
            'sinkhorn': float(a.sinkhorn), 'sinkhorn_target_mix': float(a.sinkhorn_target_mix)
        }, _fo, indent=2)
    log('V31 Bayesian typography completed; changed from pre-Bayes:', int((_raw_sub.model.values != _safe.model.values).sum()))

In [ ]:
%%writefile comp_stack.py
"""Complementary frozen-feature stack (public Apache-2.0 notebooks by prvsiyan / lucasboesen / vnton37nguyn / kirilltym,
as embedded in the public 0.6xx ATLAS notebooks). Run as a separate process; writes its raw log-probability blend."""
# ============================================================ 0. configuration
import gc, glob, json, os, pathlib, re, time
import numpy as np, pandas as pd
try:
    import torch
except ImportError:   # local smoke without torch
    torch = None
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.sparse import csr_matrix, diags, hstack, vstack
from scipy.spatial.distance import squareform
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

SMOKE = os.environ.get('NTL_SMOKE') == '1'   # local code-path check: pseudo features instead of checkpoints, CPU
T0 = time.time()
LABELS = sorted(['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
                 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
                 'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5',
                 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3'])
K = len(LABELS)
LAB2I = {l: i for i, l in enumerate(LABELS)}
FAMILY_IDX = {f: np.array([i for i, l in enumerate(LABELS) if l.startswith(f)]) for f in ('claude', 'gpt', 'gemini', 'glm')}

# ---- test-pool grouping
GROUP_THRESH, GROUP_LAYER, GROUP_MIN, GROUP_MAX = 0.60, 24, 2, 18   # mixed content + GLM-L24 distance, average linkage
STYLE_GROUP_THRESH = 0.90                                           # stylegram's own median-normalised content distance

# ---- blend weights: log-probability multipliers of the 0.563 notebook (its L-BFGS fit on prompt-grouped OOF)
W = {'stylegram': 1.516, 'stylometry_lgb': 0.063, 'ultrabert': 0.095,
     'q3_8b': 0.0, 'q25_7b': 0.0, 'q3_4b': 0.0, 'glm9b': 0.0, 'debv3l': 0.05, 'electl': 0.032, 'desk': 0.132,
     'lora24_public': 0.177, 'lora24_seed7': 0.113, 'lora24_claude': 0.177, 'lora20_q25_instyle': 0.17,
     'tokattn_q35': 0.153, 'tokattn_deb': 0.185}
TWIN_W = {'q3_8b': 1.5 * 0.219, 'q25_7b': 1.5 * 0.141, 'q3_4b': 1.5 * 0.083, 'glm9b': 1.5 * 0.110, 'debv3l': 0.25}
LORA_TWIN_W = {'lora24_public': 0.15, 'lora24_claude': 0.15}
GATE_LAM = {'claude_style': 1.5, 'claude_lora': 1.0, 'family': 0.25}  # used by confidence-aware within-family gates below
EXTRA_MIN_LABELS, EXTRA_SHARE = 12, 0.5

# ---- the addition: shared-prompt Gaussian group heads on four measured views, weight = the centred twin of the same backbone,
#      reduction / temperatures from our production fit (research/gaussian/production_weights_v8g.json, 135 training prompts)
GAUSS_HEADS = {'gauss_q3_8b_L22': dict(src='q3_8b', part='L22_mean', d=384, T1=2.351, Tg=2.306, w=TWIN_W['q3_8b']),
               'gauss_glm9b_L8': dict(src='glm9b', part='L8_mean', d=384, T1=2.722, Tg=2.596, w=TWIN_W['glm9b']),
               'gauss_q3_4b_L4': dict(src='q3_4b', part='L4_mean', d=384, T1=2.725, Tg=2.578, w=TWIN_W['q3_4b']),
               'gauss_debv3l_L24': dict(src='debv3l', part='L24_mean', d=384, T1=3.112, Tg=3.069, w=TWIN_W['debv3l'])}
GAUSS_SETTING = dict(red='wpca', se_shrink='lw', sp_shrink='lw', lam_floor=1e-4, n_exact=3, inj=True, iters=5, repel=True, sp_scale=1.0)

# ---- member specifications
LM_HEADS = [  # key, model_type, hidden size, parts of the plain head [(layer, pool)], extra parts extracted in the same pass, C
    ('glm9b', 'glm4', 4096, [(8, 'mean'), (3, 'mean')], [(GROUP_LAYER, 'mean')], 0.0025),   # first: its L24 builds the groups
    ('q3_8b', 'qwen3', 4096, [(7, 'mean'), (22, 'mean')], [], 0.0025),
    ('q25_7b', 'qwen2', 3584, [(6, 'mean'), (20, 'mean')], [], 0.0025),
    ('q3_4b', 'qwen3', 2560, [(4, 'style'), (7, 'style')], [(4, 'mean')], 0.0025)]
ENC_HEADS = [  # key, input directory name, parts, C
    ('debv3l', 'deberta-v3-large', [(24, 'mean'), (6, 'mean')], 0.005),
    ('electl', 'electra-large-discriminator', [(21, 'mean'), (24, 'style')], 0.005),
    ('desk', 'desklib-detector', [(23, 'mean'), (23, 'style')], 0.005)]
LORA_RUNS = [  # (trunk model_type, hidden, kept layers), pooled layers, [(key, checkpoint marker, norms file, anchor features file)]
    (('qwen3', 4096, 24), [2, 4, 6, 8, 12, 16, 20, 24], [
        ('lora24_public', '**/lora24/final/COMPLETE.json', 'heads.safetensors', 'anchor_features.npz'),
        ('lora24_seed7', '**/name-that-llm-train-lora24-s7/lora_out/job_final/COMPLETE.json', 'norms.safetensors', 'valid_features.npz'),
        ('lora24_claude', '**/ntl-qwen3-8b-lora-claude-head/**/final/COMPLETE.json', 'heads.safetensors', 'anchor_features.npz')]),
    (('qwen2', 3584, 20), [2, 4, 6, 8, 10, 12, 16, 20], [
        ('lora20_q25_instyle', '**/ntl-train-lora-qwen25-instyle/lora_out/job_final/COMPLETE.json', 'norms.safetensors', 'valid_features.npz')])]
LORA_MAX_LEN, LORA_BS = 447, 8
TOKATTN_HEADS = [('tokattn_q35', 'tokattn_final.pt', 'qwen3_5'), ('tokattn_deb', 'tokattn_debl23_final.pt', 'deberta')]
if os.environ.get('NTL_DROP_Q35') == '1':   # Qwen3.5-9B token-attention member: 1.7-2.7 h fixed cost on Kaggle, weight 0.153 of ~3.3
    TOKATTN_HEADS = [t for t in TOKATTN_HEADS if t[0] != 'tokattn_q35']

# ---- data
if SMOKE:
    _all = pd.read_csv(os.environ['NTL_DATA'] + '/train.csv')
    _p = _all['prompt_id'].drop_duplicates().sample(frac=1, random_state=0).tolist()
    train = _all[_all['prompt_id'].isin(_p[:30])].reset_index(drop=True)
    test = _all[_all['prompt_id'].isin(_p[30:42])].sample(frac=1, random_state=1).reset_index(drop=True)
    test = pd.concat([test[test['prompt_id'] == p].head(k) for p, k in zip(_p[30:42], [1, 1, 2, 2, 2, 3, 3, 4, 6, 9, 18, 18])]).reset_index(drop=True)
    if os.environ.get('NTL_TINY') == '1':
        test = test.head(3).reset_index(drop=True)
    SMOKE_Y = test['model'].map(LAB2I).to_numpy(dtype=int)
else:
    DATA_DIR = next(p for p in [pathlib.Path('/kaggle/input/competitions/name-that-llm'), pathlib.Path('/kaggle/input/name-that-llm')]
                    if (p / 'train.csv').exists())
    train, test = pd.read_csv(DATA_DIR / 'train.csv'), pd.read_csv(DATA_DIR / 'test.csv')
    if os.environ.get('NTL_FAKE_TEST'):   # timing probe: pool passages as a stand-in test (same rows/seed as infer.py --fake_test)
        test = pd.read_parquet(os.environ['NTL_POOL']).sample(int(os.environ['NTL_FAKE_TEST']), random_state=0)[['text']].reset_index(drop=True)
        test['id'] = [f'fake{i}' for i in range(len(test))]
test['text'] = test['text'].fillna('').astype(str)
TR, TE = train['text'].tolist(), test['text'].tolist()
y = train['model'].map(LAB2I).to_numpy(dtype=int)
PROMPT = train['prompt_id'].to_numpy(dtype=object)
TINY = len(TE) < 50   # the interactive 3-row test: exercise the group paths on training rows so the hidden rerun cannot hit untested code
DEV = 'cuda' if torch is not None and torch.cuda.is_available() else 'cpu'
print(f'train {train.shape}  test {test.shape}  gpus {torch.cuda.device_count() if torch is not None else 0}  smoke {SMOKE}', flush=True)

LOGIT = np.zeros((len(TE), K))           # the running log-probability blend


def add_member(key, prob, rows=None, weight=None):
    """Add weight * log(prob) to the blend (all rows, or the given row subset)."""
    w = W[key] if weight is None else weight
    lp = np.log(np.clip(prob, 1e-9, 1.0))
    if rows is None:
        LOGIT[:] += w * lp
    else:
        LOGIT[rows] += w * lp
    print(f'  + {key:20s} weight {w:.3f} on {len(TE) if rows is None else int(np.sum(rows))} rows  ({time.time() - T0:.0f}s)', flush=True)


def softmax_rows(S):
    S = S - S.max(1, keepdims=True)
    P = np.exp(S)
    return P / P.sum(1, keepdims=True)


def full_prob(P_part, classes):
    """Expand a (n, len(classes)) probability block to (n, K) on the global label order."""
    P = np.zeros((len(P_part), K))
    P[:, classes] = P_part
    return P


def group_centre_dense(X, groups):
    X = np.asarray(X, dtype=np.float32).copy()
    for g in np.unique(groups):
        i = groups == g
        X[i] -= X[i].mean(0)
    return X


def self_test_rows():
    """Training rows of the first three prompts, grouped by their true prompt (used only when the test file is tiny)."""
    idx = np.where(np.isin(PROMPT, pd.unique(PROMPT)[:3]))[0]
    return idx, np.unique(PROMPT[idx], return_inverse=True)[1]

# ============================================================ 1. grouping the test pool
def average_linkage_groups(D, thresh):
    """Cluster ids (1-based) from a square distance matrix, average linkage, cut at thresh."""
    if len(D) < 2:
        return np.ones(len(D), dtype=int)
    D = np.asarray(D, dtype=np.float64).copy()
    np.fill_diagonal(D, 0.0)
    return fcluster(linkage(squareform(D, checks=False), method='average'), thresh, criterion='distance')


def content_cosine_distance(texts, max_features=None):
    X = TfidfVectorizer(stop_words='english', sublinear_tf=True, max_features=max_features, dtype=np.float32).fit_transform(texts)
    return np.clip(1.0 - (X @ X.T).toarray(), 0.0, 1.0)


def mixed_groups(texts, E, thresh=GROUP_THRESH):
    """Average of the content-word TF-IDF cosine distance and the cosine distance of mean-centred, L2-normalised embeddings E."""
    if len(texts) < 2:
        return np.ones(len(texts), dtype=int)
    Z = np.asarray(E, dtype=np.float64)
    Z = Z - Z.mean(0)
    Z /= np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12
    D = 0.5 * (content_cosine_distance(texts) + np.clip(1.0 - Z @ Z.T, 0.0, 2.0))
    return average_linkage_groups(D, thresh)


def normalised_content_groups(texts, thresh=STYLE_GROUP_THRESH):
    """The stylegram's grouping: content distance divided by its median off-diagonal value, cut at thresh."""
    if len(texts) < 2:
        return np.ones(len(texts), dtype=int)
    D = content_cosine_distance(texts, max_features=100_000)
    med = max(float(np.median(D[np.triu_indices(len(texts), 1)])), 1e-6)
    return average_linkage_groups(D / med, thresh)


def eligible(groups, lo=GROUP_MIN, hi=GROUP_MAX):
    size = np.bincount(groups)[groups]
    return (size >= lo) & (size <= hi)


def describe_groups(name, groups):
    size = np.bincount(groups)[groups]
    print(f'{name}: {len(groups)} rows | singletons {int((size == 1).sum())}  2-3: {int(((size >= 2) & (size <= 3)).sum())}'
          f'  4-18: {int(((size >= 4) & (size <= 18)).sum())}  >18: {int((size > 18).sum())}', flush=True)


G_TE, GROUP_ROWS = None, None   # set after the GLM pass (section 4)

# ============================================================ 2. style n-grams + extra passages + gates
FUNCTION_WORDS = frozenset("a an the and or but if while although because as so than that which who whom whose this these those it its "
                           "they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with "
                           "without for of by over under between through into about is are was were be been being have has had do does did "
                           "can could will would shall should may might must not no nor all any some each every both either neither other "
                           "another such only also just very too more most less least here there when where why how what whether then now "
                           "still yet even much many few little".split())
TOKEN_RE = re.compile(r"\w+|[^\w\s]")


def shape_view(text):
    return re.sub(r"[^\W\d_]+", lambda m: 'X' if m[0].isupper() else 'x', text)


def function_view(text):
    return ' '.join(t.lower() if (t.lower() in FUNCTION_WORDS or not t.isalnum()) else 'CONTENT' for t in TOKEN_RE.findall(text))


class StyleGrams:
    """Four stacked TF-IDF views: char 3-5, word 1-3, shape (char 2-6 of the x/X text), function-word (word 1-3 of the masked text)."""
    VIEWS = [('char', str, (3, 5), 250_000, 1.0), ('word', str, (1, 3), 180_000, 0.7),
             ('char', shape_view, (2, 6), 60_000, 0.5), ('word', function_view, (1, 3), 50_000, 0.5)]

    def fit(self, texts):
        self.vecs = []
        for analyzer, view, ngr, maxf, w in self.VIEWS:
            opts = dict(analyzer=analyzer, ngram_range=ngr, lowercase=False, sublinear_tf=True, min_df=2, max_features=maxf, dtype=np.float64)
            if analyzer == 'word':
                opts['token_pattern'] = r"(?u)\b\w+\b|[^\w\s]"
            self.vecs.append((TfidfVectorizer(**opts).fit([view(t) for t in texts]), view, w))
        return self

    def transform(self, texts):
        return hstack([v.transform([view(t) for t in texts]) * w for v, view, w in self.vecs], format='csr')


def sparse_group_centre(X, groups, lo=GROUP_MIN, hi=GROUP_MAX):
    """Subtract each group's mean row (sparse); groups outside [lo, hi] are left as they are."""
    X = csr_matrix(X)
    _, inv, cnt = np.unique(np.asarray(groups), return_inverse=True, return_counts=True)
    keep = np.flatnonzero((cnt >= lo) & (cnt <= hi))
    if not len(keep):
        return X.copy()
    col = np.full(len(cnt), -1); col[keep] = np.arange(len(keep))
    rows = np.flatnonzero(col[inv] >= 0)
    M = csr_matrix((np.ones(len(rows)), (rows, col[inv][rows])), shape=(X.shape[0], len(keep)))
    means = diags(1.0 / cnt[keep]) @ (M.T @ X)
    return (X - M @ means).tocsr()


def svc_probs(svc, X):
    return full_prob(softmax_rows(svc.decision_function(X)), svc.classes_)


# ---- public in-style passages (prompt-bank protocol), one per (prompt, label); prompts answered by >= EXTRA_MIN_LABELS labels
def load_extra_passages():
    sources = [('**/ntl_extra_passages_claude.csv', 'model', None), ('**/gen_passages.csv', 'model', None),
               ('**/ntl_generations.csv', 'label', 'finish_reason'), ('**/ntl_proxy_generations.csv', 'label', None),
               ('**/ntl_generated*.csv', 'model', 'finish'), ('**/passages_clean.csv', 'model', None)]
    parts = []
    for pattern, label_col, finish_col in sources:
        for f in sorted(glob.glob('/kaggle/input/' + pattern, recursive=True)):
            d = pd.read_csv(f)
            if finish_col and finish_col in d:
                d = d[d[finish_col].eq('stop')]
            d = d.rename(columns={label_col: 'model'})[['prompt_id', 'model', 'text']].copy()
            d['prompt_id'] = ('pv_' if f.endswith('passages_clean.csv') else '') + d['prompt_id'].astype(str)   # prvsiyan's prompts are a different bank
            parts.append(d)
    if not parts:
        return pd.DataFrame(columns=['prompt_id', 'model', 'text'])
    x = pd.concat(parts, ignore_index=True)
    x = x[x['model'].isin(LABELS)].copy()
    x['text'] = x['text'].astype(str).map(lambda t: re.sub(r'\s+', ' ', re.sub(r'^\*\*(.*)\*\*$', r'\1', t.strip())).strip())
    x = x[x['text'].str.split().str.len().between(30, 300)].drop_duplicates(subset=['model', 'text'])
    x = x.drop_duplicates(subset=['prompt_id', 'model'])
    x = x[x.groupby('prompt_id')['model'].transform('nunique') >= EXTRA_MIN_LABELS].reset_index(drop=True)
    return x


EXTRA = load_extra_passages()
EX_T, EX_Y, EX_G = EXTRA['text'].tolist(), EXTRA['model'].map(LAB2I).to_numpy(dtype=int), EXTRA['prompt_id'].to_numpy(dtype=object)
print(f'extra passages: {len(EXTRA)} rows, {EXTRA["prompt_id"].nunique()} prompts', flush=True)


def extra_sample_weights():
    """Per class: original rows keep (1 - EXTRA_SHARE) of the class weight, the class's extra rows share EXTRA_SHARE of it."""
    n_tr, n_ex = np.bincount(y, minlength=K).astype(float), np.bincount(EX_Y, minlength=K).astype(float)
    w_tr = np.where(n_ex[y] > 0, 1.0 - EXTRA_SHARE, 1.0)
    w_ex = EXTRA_SHARE * n_tr[EX_Y] / np.maximum(n_ex[EX_Y], 1.0)
    return np.concatenate([w_tr, w_ex])


grams = StyleGrams().fit(TR + EX_T)                       # vocabulary and idf from train + extra
X_tr, X_ex, X_te = grams.transform(TR), grams.transform(EX_T) if len(EX_T) else None, grams.transform(TE)


def fit_svc(X, labels, C=0.5, **kw):
    return LinearSVC(C=C, random_state=42, max_iter=4000).fit(X, labels, **kw)


if X_ex is not None:
    svc_plain = fit_svc(vstack([X_tr, X_ex]).tocsr(), np.concatenate([y, EX_Y]), sample_weight=extra_sample_weights())
else:
    svc_plain = fit_svc(X_tr, y)
P_style = svc_probs(svc_plain, X_te)

# centred twin: prompt-centred training rows (extra rows centred within their own prompts) -> inferred test groups
X_tr_c = sparse_group_centre(X_tr, PROMPT)
if X_ex is not None:
    svc_centred = fit_svc(vstack([X_tr_c, sparse_group_centre(X_ex, EX_G, GROUP_MIN, 10 ** 6)]).tocsr(), np.concatenate([y, EX_Y]),
                          sample_weight=extra_sample_weights())
else:
    svc_centred = fit_svc(X_tr_c, y)
G_style = normalised_content_groups(TE)
rows_style = eligible(G_style)
describe_groups('stylegram groups', G_style)
if rows_style.any():
    P_style[rows_style] = svc_probs(svc_centred, sparse_group_centre(X_te[rows_style], G_style[rows_style]))
if TINY:   # exercise the centred path on training rows grouped by their true prompt (output discarded)
    st_idx, st_g = self_test_rows()
    acc = np.mean(svc_centred.classes_[svc_centred.decision_function(sparse_group_centre(X_tr[st_idx], st_g)).argmax(1)] == y[st_idx])
    print(f'  centred stylegram self-test on {len(st_idx)} training rows (seen in training): acc {acc:.2f}', flush=True)
add_member('stylegram', P_style)
del X_tr_c, svc_plain, svc_centred, P_style


# ---- gates: style SVCs that only re-rank inside one family after the blend
def family_gate(texts, labels, classes, C, sample_weight=None, balanced=False):
    g = StyleGrams().fit(texts)
    svc = LinearSVC(C=C, class_weight='balanced' if balanced else None, random_state=42, max_iter=4000).fit(
        g.transform(texts), labels, sample_weight=sample_weight)
    assert list(svc.classes_) == list(classes)
    d = svc.decision_function(g.transform(TE))
    return np.column_stack([-d, d]) if len(classes) == 2 else d


GATES = {}
api_claude = sorted(glob.glob('/kaggle/input/**/ntl_extra_passages_claude.csv', recursive=True))
if api_claude:
    api = pd.read_csv(api_claude[0])
    m = np.isin(y, FAMILY_IDX['claude'])
    GATES['claude'] = family_gate([t for t, k in zip(TR, m) if k] + api['text'].astype(str).tolist(),
                                  np.concatenate([y[m], api['model'].map(LAB2I).to_numpy(dtype=int)]), FAMILY_IDX['claude'], 0.5)
    print(f'claude gate: {int(m.sum())} official + {len(api)} API rows', flush=True)
    del api
else:
    print('Claude API passages not attached: Claude gate off', flush=True)
for fam, C in [('gpt', 1.0), ('gemini', 1.0), ('glm', 0.5)]:
    m, me = np.isin(y, FAMILY_IDX[fam]), np.isin(EX_Y, FAMILY_IDX[fam])
    GATES[fam] = family_gate([t for t, k in zip(TR, m) if k] + [t for t, k in zip(EX_T, me) if k], np.concatenate([y[m], EX_Y[me]]),
                             FAMILY_IDX[fam], C, sample_weight=np.concatenate([np.ones(int(m.sum())), np.full(int(me.sum()), 0.25)]), balanced=True)
    print(f'{fam} gate: {int(m.sum())} official + {int(me.sum())} extra rows', flush=True)
del grams, X_tr, X_ex, X_te

# ============================================================ 3. stylometric features + LightGBM
PUNCT = list(',.;:!?()[]"\'-') + ['—', '–', '‘', '’', '“', '”', '…', '*', '/', '%']
STOPS = sorted(ENGLISH_STOP_WORDS)
SENT_RE, WORD_RE = re.compile(r'(?<=[.!?])\s+'), re.compile(r"[A-Za-z’']+")


def stylometry(t):
    words = WORD_RE.findall(t)
    low = [w.lower() for w in words]
    nw, nc = max(len(words), 1), max(len(t), 1)
    sents = [s for s in SENT_RE.split(t.strip()) if s]
    sl = np.array([len(WORD_RE.findall(s)) for s in sents] or [0], dtype=float)
    wl = np.array([len(w) for w in words] or [0], dtype=float)
    f = [np.log1p(nw), len(sents), sl.mean(), sl.std(), sl.max(), sl.min(), wl.mean(), wl.std(), (wl >= 9).mean(), len(set(low)) / nw,
         sum(c.isupper() for c in t) / nc, sum(c.isdigit() for c in t) / nc, sum(ord(c) > 127 for c in t) / nc,
         t.count(', ') / nw, t.count(' and ') / nw, t.count(' but ') / nw,
         sum(1 for s in sents if s[:1].isupper() and s.split()[0].lower() in ('this', 'these', 'it', 'they')) / max(len(sents), 1),
         sum(w.endswith('ly') for w in words) / nw, sum(w.endswith('ing') for w in words) / nw, sum(w.endswith('tion') for w in words) / nw]
    f += [t.count(q) / nw for q in PUNCT]
    counts = pd.Series(low).value_counts() if low else pd.Series(dtype=float)
    f += [counts.get(s, 0) / nw for s in STOPS]
    return f


F_tr, F_te = (np.array([stylometry(t) for t in txt], dtype=np.float32) for txt in (TR, TE))
try:
    import lightgbm as lgb
    booster = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.03, num_leaves=15, min_child_samples=10, subsample=0.8, subsample_freq=1,
                                 colsample_bytree=0.5, reg_lambda=1.0, random_state=0, verbose=-1, n_jobs=4).fit(F_tr, y)
    add_member('stylometry_lgb', full_prob(booster.predict_proba(F_te), booster.classes_))
    del booster
except ImportError:
    assert SMOKE, 'lightgbm missing'
    print('smoke: lightgbm unavailable, stylometry member skipped', flush=True)
del F_tr, F_te

# ============================================================ 4b. shared-prompt Gaussian group head
# Verbatim core of our research tool tools/group_gaussian.py (sha256 8a42427b77a37f56...), numpy only.
# Model per dense view, in a reduced space z (standardise -> within-prompt-residual PCA, fitted on training rows):
#   z_i = mu_{y_i} + p_g + e_i,  p_g ~ N(0, S_p) shared by every passage of prompt group g,  e_i ~ N(0, S_e).
# A joint diagonalisation makes the dimensions independent; for a detected group of n passages the exact joint
# log-likelihood of a label tuple is sum_i a_i(c_i) + sum_{i<j} b_ij(c_i, c_j). Groups of 2-3 are decoded exactly over
# all label tuples with the at-most-once constraint; larger groups by a cavity EM with the finite-population-corrected
# prompt-effect estimate lam / (1 + (n-1) lam); singletons integrate the prompt effect out (never used here).
import itertools
K = 18
DEFAULT = {"red": "wpca", "d": 256, "se_shrink": "lw", "sp_shrink": "lw", "lam_floor": 1e-4, "n_exact": 3, "inj": True,
           "iters": 5, "repel": True, "sp_scale": 1.0}

# =================================================================== primitives ====
def _lse(A, axis=None, keepdims=False):
    m = np.max(A, axis=axis, keepdims=True); m = np.where(np.isfinite(m), m, 0.0)
    out = np.log(np.sum(np.exp(A - m), axis=axis, keepdims=True)) + m
    if keepdims: return out
    return np.squeeze(out, axis=axis) if axis is not None else float(out.item())

def log_softmax(L): return L - _lse(L, axis=1, keepdims=True)

def _lw_shrinkage(X):
    """Ledoit-Wolf shrinkage coefficient for rows X that are already centred (sklearn formula, assume_centered)."""
    from sklearn.covariance import ledoit_wolf_shrinkage
    return float(ledoit_wolf_shrinkage(X, assume_centered=True))

def _shrunk(S, a):
    d = S.shape[0]; return (1 - a) * S + a * (np.trace(S) / d) * np.eye(d)

def _two_way(Z, y, pinv, G, iters=4):
    """additive fit Z = mu[y] + P[prompt] + E by alternating means (balanced design -> converges in 1-2 rounds)."""
    P = np.zeros((G, Z.shape[1])); cnt_p = np.bincount(pinv, minlength=G).astype(float)
    cls = np.unique(y); mu = np.zeros((K, Z.shape[1]))
    for _ in range(iters):
        R = Z - P[pinv]
        for c in cls: mu[c] = R[y == c].mean(0)
        R = Z - mu[y]; P = np.zeros_like(P); np.add.at(P, pinv, R); P /= cnt_p[:, None]
    return mu, P, cnt_p

def fit(X, y, prompts, cfg=None):
    """fit the shared-prompt Gaussian on training rows. X (N, D) raw features, y (N,) labels in 0..17, prompts (N,) group ids.
    Returns a params dict (numpy arrays + floats); temperatures default to 1 (set by the caller's cross-fit)."""
    c = dict(DEFAULT); c.update(cfg or {})
    X = np.asarray(X, dtype=np.float64); y = np.asarray(y); N, D = X.shape
    pu, pinv = np.unique(np.asarray(prompts), return_inverse=True); G = len(pu)
    mx = X.mean(0); sx = X.std(0); sx = np.where(sx > 1e-8, sx, 1.0); Xs = (X - mx) / sx
    if c["red"] == "none":
        V = np.eye(D)
    else:
        if c["red"] == "wpca":
            Pm = np.zeros((G, D)); np.add.at(Pm, pinv, Xs); Pm /= np.bincount(pinv)[:, None]; R = Xs - Pm[pinv]
        elif c["red"] == "pca": R = Xs
        else: raise ValueError(c["red"])
        d = int(min(c["d"], N - G - 1, D))
        # economy SVD through the (N x N) Gram matrix (N << D for the real views)
        if N < D:
            w, U = np.linalg.eigh(R @ R.T); o = np.argsort(w)[::-1][:d]; w, U = np.clip(w[o], 1e-12, None), U[:, o]
            V = (R.T @ U) / np.sqrt(w)
        else:
            _, _, Vt = np.linalg.svd(R, full_matrices=False); V = Vt[:d].T
    Z = Xs @ V; d = Z.shape[1]
    mu, P, ng = _two_way(Z, y, pinv, G)
    E = Z - mu[y] - P[pinv]; dof = max(N - G - len(np.unique(y)) + 1, 1)
    Se_raw = E.T @ E / dof
    ae = _lw_shrinkage(E) if c["se_shrink"] == "lw" else float(c["se_shrink"]); Se = _shrunk(Se_raw, ae)
    Pc = P - P.mean(0); Sp_raw = Pc.T @ Pc / max(G - 1, 1) - np.mean(1.0 / ng) * Se_raw
    ap = _lw_shrinkage(Pc) if c["sp_shrink"] == "lw" else float(c["sp_shrink"])
    tr = max(np.trace(Sp_raw), 1e-8 * np.trace(Se)); Sp = (1 - ap) * Sp_raw + ap * (tr / d) * np.eye(d); Sp *= c["sp_scale"]
    # joint diagonalisation: A Se A' = I, A Sp A' = diag(lam)
    L = np.linalg.cholesky(Se); Li = np.linalg.inv(L); M = Li @ Sp @ Li.T; M = 0.5 * (M + M.T)
    lam, Q = np.linalg.eigh(M); lam = np.clip(lam, c["lam_floor"], None); A = Q.T @ Li
    W = V @ A.T
    return {"mx": mx, "sx": sx, "W": W, "m": mu @ A.T, "lam": lam, "T1": 1.0, "Tg": 1.0,
            "info": {"d": int(d), "a_e": round(ae, 4), "a_p": round(ap, 4), "lam_pct": np.round(np.percentile(lam, [5, 50, 95, 100]), 3).tolist()}, "cfg": c}

def transform(X, prm): return ((np.asarray(X, dtype=np.float64) - prm["mx"]) / prm["sx"]) @ prm["W"]

_MASK = {}
def _inj_mask(n, C):
    if (n, C) not in _MASK:
        grids = np.meshgrid(*[np.arange(C)] * n, indexing="ij"); bad = np.zeros([C] * n, bool)
        for i, j in itertools.combinations(range(n), 2): bad |= grids[i] == grids[j]
        _MASK[(n, C)] = bad
    return _MASK[(n, C)]

def singleton_ll(U, m, lam):
    """(n, C) log N(u; m_c, I + diag lam) up to a constant (p integrated out)."""
    R = U[:, None, :] - m[None]; return -0.5 * (R ** 2 / (1 + lam)).sum(-1)

def joint_terms(U, m, lam):
    """a (n, C) and B (n, n, C, C) of the exact joint log-likelihood of one group (constant in c dropped)."""
    n = len(U); gam = lam / (1 + n * lam); R = U[:, None, :] - m[None]
    a = -0.5 * (R ** 2 * (1 - gam)).sum(-1); G = R * np.sqrt(gam); B = np.einsum("icd,jed->ijce", G, G)
    return a, B

def joint_table(a, B):
    """full tuple table Q[c_1, ..., c_n] = sum_i a_i + sum_{i<j} B_ij."""
    n, C = a.shape; Q = np.zeros([C] * n)
    for i in range(n):
        sh = [1] * n; sh[i] = C; Q = Q + a[i].reshape(sh)
    for i, j in itertools.combinations(range(n), 2):
        sh = [1] * n; sh[i] = C; sh[j] = C; Q = Q + B[i, j].reshape(sh)
    return Q

def exact_marginals(Q, T=1.0, inj=True):
    """per-passage log marginal posteriors (n, C) from a tuple table Q (unnormalised log joint), tempered by 1/T."""
    n = Q.ndim; C = Q.shape[0]; Qt = Q / T
    if inj and n >= 2: Qt = np.where(_inj_mask(n, C), -np.inf, Qt)
    lz = _lse(Qt); out = np.empty((n, C))
    for i in range(n):
        ax = tuple(k for k in range(n) if k != i); out[i] = (_lse(Qt, axis=ax) if ax else Qt) - lz
    return out

def em_marginals(U, m, lam, T=1.0, iters=5, repel=False, q0=None):
    """cavity EM for one group (any n >= 2): returns (n, C) log q."""
    n = len(U); v = lam / (1 + (n - 1) * lam)
    L = singleton_ll(U, m, lam) / T if q0 is None else np.log(np.clip(q0, 1e-300, None))
    q = np.exp(log_softmax(L))
    for _ in range(iters):
        s = U - q @ m; S = s.sum(0); Pm = (S[None] - s) * v                  # cavity prompt-effect means (n, d)
        R = U[:, None, :] - Pm[:, None, :] - m[None]; L = -0.5 * (R ** 2 / (1 + v)).sum(-1) / T
        if repel:   # mean-field at-most-once: discourage classes the other members already hold
            lo = np.log(np.clip(1 - q, 1e-6, None)); L = L + (lo.sum(0)[None] - lo)
        q = np.exp(log_softmax(L))
    return log_softmax(L)

LAM_DIFF = 1e6
def lam_g(prm, cfg=None):
    """lam used for groups: the fitted one, or ~infinity in the paired-difference ablation (cfg 'diff'): then the joint likelihood
    only sees within-group differences (x_i - x_j around mu_a - mu_b with covariance 2 S_e; triples: centred observations vs
    centred prototypes) and the prompt term cancels exactly."""
    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {})
    return np.full_like(prm["lam"], LAM_DIFF) if c.get("diff") else prm["lam"]

def group_logpost(U, prm, T1=None, Tg=None, cfg=None):
    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {}); n = len(U); lg = lam_g(prm, cfg)
    T1 = prm["T1"] if T1 is None else T1; Tg = prm["Tg"] if Tg is None else Tg
    if n == 1: return log_softmax(singleton_ll(U, prm["m"], prm["lam"]) / T1)
    if n <= c["n_exact"]:
        a, B = joint_terms(U, prm["m"], lg); return exact_marginals(joint_table(a, B), Tg, c["inj"])
    return em_marginals(U, prm["m"], lg, Tg, c["iters"], c["repel"])

def scores(X_pool, lab, prm, T1=None, Tg=None, cfg=None, U=None):
    """(n, 18) log-probabilities for every pool row; lab = detected group labels of the pool rows (singletons allowed)."""
    U = transform(X_pool, prm) if U is None else U; lab = np.asarray(lab); n = len(U); out = np.empty((n, prm["m"].shape[0]))
    sizes = np.bincount(lab); s1 = sizes[lab] == 1
    T1 = prm["T1"] if T1 is None else T1
    if s1.any(): out[s1] = log_softmax(singleton_ll(U[s1], prm["m"], prm["lam"]) / T1)
    for g in np.where(sizes >= 2)[0]:
        ii = np.where(lab == g)[0]; out[ii] = group_logpost(U[ii], prm, T1, Tg, cfg)
    return out

# ============================================================ 4. frozen models: plain heads, centred twins, Gaussian group heads
if not SMOKE:
    from transformers import AutoConfig, AutoModel, AutoTokenizer
CACHED_PARTS = {}   # key -> {part name: (train features, test features)}


def find_checkpoint(model_type, hidden=None, dir_name=None):
    """A checkpoint directory under /kaggle/input by model_type (+ hidden size) or by directory name."""
    pattern = f'/kaggle/input/**/{dir_name}/config.json' if dir_name else '/kaggle/input/**/config.json'
    for cfg in sorted(glob.glob(pattern, recursive=True)):
        try:
            c = json.load(open(cfg))
        except Exception:
            continue
        if dir_name or (c.get('model_type') == model_type and (hidden is None or c.get('hidden_size') == hidden) and glob.glob(os.path.join(os.path.dirname(cfg), '*.safetensors'))):
            return os.path.dirname(cfg)
    return None


def load_truncated_decoder(path, keep, dtype):
    cfg = AutoConfig.from_pretrained(path)
    cfg.num_hidden_layers = keep
    if getattr(cfg, 'layer_types', None):
        cfg.layer_types = cfg.layer_types[:keep]
    kw = dict(config=cfg, dtype=dtype)
    if DEV == 'cuda':
        kw['device_map'] = 'auto'
    return AutoModel.from_pretrained(path, **kw).eval()


def style_token_mask(tok):
    """Vocabulary mask of 'style' tokens: the decoded piece is an English stop word or pure punctuation."""
    pieces = [tok.decode([i]) for i in range(len(tok))]
    return torch.tensor([(p.strip().lower() in ENGLISH_STOP_WORDS) or bool(re.fullmatch(r'\s*[^\w\s]+\s*', p)) for p in pieces])


@torch.no_grad()
def pooled_states(model, tok, texts, parts, decoder, bs, max_len=512):
    """{'L{layer}_{pool}': (n, hidden)} masked mean pools of hidden_states[layer] (all tokens, or style tokens only)."""
    H = model.config.hidden_size
    out = {f'L{L}_{pool}': np.zeros((len(texts), H), dtype=np.float32) for L, pool in parts}
    smask = style_token_mask(tok) if any(pool == 'style' for _, pool in parts) else None
    dev = next(model.parameters()).device
    order = np.argsort([len(t) for t in texts])
    retried = 0
    for b0 in range(0, len(texts), bs):
        idx = order[b0:b0 + bs]
        if decoder:
            batch = [(tok.bos_token or '') + texts[i] for i in idx]
            enc = tok(batch, return_tensors='pt', padding=True, truncation=True, max_length=max_len, add_special_tokens=False).to(dev)
        else:
            enc = tok([texts[i] for i in idx], return_tensors='pt', padding=True, truncation=True, max_length=max_len).to(dev)
        hs = model(**enc, output_hidden_states=True).hidden_states
        if decoder and not all(torch.isfinite(hs[L]).all() for L, _ in parts):   # fp16 overflow: redo the batch in bf16
            retried += 1
            model.to(torch.bfloat16)
            hs = model(**enc, output_hidden_states=True).hidden_states
        m_all = enc['attention_mask'].float()
        for L, pool in parts:
            m = m_all * smask.to(m_all.device)[enc['input_ids']].float() if pool == 'style' else m_all
            m = m.unsqueeze(-1).to(hs[L].device)
            out[f'L{L}_{pool}'][idx] = ((hs[L].float() * m).sum(1) / m.sum(1).clamp(min=1)).cpu().numpy()
        del hs, enc
    if retried:
        print(f'  {retried} batches redone in bf16', flush=True)
    return {k: np.nan_to_num(v) for k, v in out.items()}


def pseudo_parts(parts, dim=48):
    """SMOKE only: deterministic stand-ins for the model passes (TF-IDF SVD of the texts), same downstream code."""
    from sklearn.decomposition import TruncatedSVD
    tf = TfidfVectorizer(analyzer='char', ngram_range=(2, 4), sublinear_tf=True, min_df=2).fit(TR + TE)
    out = {}
    for j, (L, pool) in enumerate(parts):
        svd = TruncatedSVD(dim, random_state=L + 7 * j).fit(tf.transform(TR))
        out[f'L{L}_{pool}'] = (svd.transform(tf.transform(TR)).astype(np.float32), svd.transform(tf.transform(TE)).astype(np.float32))
    return out


def extract_parts(key, parts, loader):
    """Train and test pools for every requested part; loader() returns (model, tok, decoder flag, batch size)."""
    if SMOKE:
        return pseudo_parts(parts)
    model, tok, decoder, bs = loader()
    tr, te = pooled_states(model, tok, TR, parts, decoder, bs), pooled_states(model, tok, TE, parts, decoder, bs)
    del model, tok
    gc.collect()
    if DEV == 'cuda':
        torch.cuda.empty_cache()
    return {k: (tr[k], te[k]) for k in tr}


def decoder_loader(model_type, hidden, keep):
    def load():
        path = find_checkpoint(model_type, hidden)
        assert path, f'no {model_type} checkpoint with hidden size {hidden} under /kaggle/input'
        tok = AutoTokenizer.from_pretrained(path)
        tok.padding_side = 'right'
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        return load_truncated_decoder(path, keep, torch.float16 if DEV == 'cuda' else torch.float32), tok, True, 8
    return load


def encoder_loader(dir_name):
    def load():
        path = find_checkpoint(None, dir_name=dir_name)
        assert path, f'encoder directory {dir_name} not found under /kaggle/input'
        return AutoModel.from_pretrained(path, dtype=torch.float32).to(DEV).eval(), AutoTokenizer.from_pretrained(path), False, 16
    return load


def concat_parts(cache, parts):
    return (np.concatenate([cache[f'L{L}_{p}'][0] for L, p in parts], 1), np.concatenate([cache[f'L{L}_{p}'][1] for L, p in parts], 1))


def plain_and_twin(key, Etr, Ete, C):
    """Plain head (test-side standardisation) for the blend; centred twin evidence on grouped rows."""
    if W[key] > 0:
        sc = StandardScaler().fit(Etr)
        clf = LogisticRegression(C=C, max_iter=5000).fit(sc.transform(Etr), y)
        Xte = (Ete - Ete.mean(0)) / (Ete.std(0) + 1e-6) if len(Ete) >= 2 else sc.transform(Ete)
        add_member(key, full_prob(clf.predict_proba(Xte), clf.classes_))
    else:
        print(f'  {key}: plain head has weight 0 in the 0.563 blend, not fitted', flush=True)
    if key in TWIN_W:
        csc = StandardScaler().fit(group_centre_dense(Etr, PROMPT))
        twin = LogisticRegression(C=C, max_iter=5000).fit(csc.transform(group_centre_dense(Etr, PROMPT)), y)
        if GROUP_ROWS.any():
            pc = twin.predict_proba(csc.transform(group_centre_dense(Ete[GROUP_ROWS], G_TE[GROUP_ROWS])))
            add_member(key + '_twin', full_prob(pc, twin.classes_), rows=GROUP_ROWS, weight=TWIN_W[key])
        if TINY:
            st_idx, st_g = self_test_rows()
            ps = twin.predict_proba(csc.transform(group_centre_dense(Etr[st_idx], st_g)))
            print(f'  {key} twin self-test on {len(st_idx)} training rows (seen in training): acc {np.mean(twin.classes_[ps.argmax(1)] == y[st_idx]):.2f}', flush=True)


def gaussian_group_head(name, spec, Xtr, Xte):
    """Our addition: fit the shared-prompt Gaussian on the training prompts, score the grouped test rows jointly within their group."""
    prm = fit(Xtr, y, PROMPT, dict(GAUSS_SETTING, d=spec['d']))
    prm['T1'], prm['Tg'] = spec['T1'], spec['Tg']
    print(f'  {name}: d {prm["info"]["d"]}, shrinkage e/p {prm["info"]["a_e"]}/{prm["info"]["a_p"]}, lam pct {prm["info"]["lam_pct"]}', flush=True)
    if GROUP_ROWS.any():
        LP = scores(Xte[GROUP_ROWS], np.unique(G_TE[GROUP_ROWS], return_inverse=True)[1], prm)
        add_member(name, np.exp(LP), rows=GROUP_ROWS, weight=spec['w'])
    if TINY:
        st_idx, st_g = self_test_rows()
        LP = scores(Xtr[st_idx], st_g, prm)
        print(f'  {name} self-test on {len(st_idx)} training rows (seen in training): acc {np.mean(LP.argmax(1) == y[st_idx]):.2f}', flush=True)


for key, model_type, hidden, parts, extra_parts, C in LM_HEADS:
    print(f'[{key}] {model_type} hidden {hidden} parts {parts + extra_parts}', flush=True)
    cache = extract_parts(key, parts + extra_parts, decoder_loader(model_type, hidden, max(L for L, _ in parts + extra_parts) + 1))
    if G_TE is None:   # the GLM pass comes first: its layer-24 pool defines the test groups
        G_TE = mixed_groups(TE, cache[f'L{GROUP_LAYER}_mean'][1])
        GROUP_ROWS = eligible(G_TE)
        describe_groups('test groups (content + GLM L24)', G_TE)
        if SMOKE:
            same_g, same_p = G_TE[:, None] == G_TE[None, :], test['prompt_id'].to_numpy(dtype=object)[:, None] == test['prompt_id'].to_numpy(dtype=object)[None, :]
            off = ~np.eye(len(TE), dtype=bool)
            print(f'  smoke grouping: pair precision {(same_g & same_p & off).sum() / max((same_g & off).sum(), 1):.2f} '
                  f'recall {(same_g & same_p & off).sum() / max((same_p & off).sum(), 1):.2f}', flush=True)
    CACHED_PARTS[key] = cache
    plain_and_twin(key, *concat_parts(cache, parts), C)
    for name, spec in GAUSS_HEADS.items():
        if spec['src'] == key:
            gaussian_group_head(name, spec, *cache[spec['part']])
    del cache
    CACHED_PARTS.pop(key)
    gc.collect()

for key, dir_name, parts, C in ENC_HEADS:
    print(f'[{key}] {dir_name} parts {parts}', flush=True)
    cache = extract_parts(key, parts, encoder_loader(dir_name))
    plain_and_twin(key, *concat_parts(cache, parts), C)
    for name, spec in GAUSS_HEADS.items():
        if spec['src'] == key:
            gaussian_group_head(name, spec, *cache[spec['part']])
    del cache
    gc.collect()

# ============================================================ 5. LoRA classifiers (inference only)
@torch.no_grad()
def score_lora_checkpoints(trunk_spec, pool_layers, checkpoints):
    """One truncated trunk, adapters swapped in turn. Plain readout -> blend member; centred readout -> twin on grouped rows;
    within-Claude readout -> Claude gate."""
    import importlib.util as iu
    _find = iu.find_spec   # Kaggle's torchao build is rejected by peft; hide it (unused here)
    iu.find_spec = lambda name, *a, **k: None if name.split('.')[0] == 'torchao' else _find(name, *a, **k)
    from peft import PeftModel
    import peft.import_utils as pi
    pi.is_torchao_available = lambda *a, **k: False
    from safetensors.torch import load_file
    model_type, H, keep = trunk_spec
    base = find_checkpoint(model_type, H)
    assert base, f'no {model_type} base checkpoint under /kaggle/input'
    cfg = AutoConfig.from_pretrained(base)
    cfg.num_hidden_layers, cfg.use_cache = keep, False
    if getattr(cfg, 'layer_types', None) is not None:
        cfg.layer_types = cfg.layer_types[:keep]
    trunk = AutoModel.from_pretrained(base, config=cfg, dtype=torch.float16, device_map='auto').eval()
    for key, marker, norms_file, anchor_file in checkpoints:
        hits = sorted(glob.glob('/kaggle/input/' + marker, recursive=True))
        if not hits:   # upstream training kernel no longer public: drop this member, keep the rest of the stack
            print(f'  SKIP {key}: checkpoint not attached ({marker})', flush=True); continue
        ck = pathlib.Path(os.path.dirname(hits[0]))
        print(f'[{key}] {ck}', flush=True)
        tok = AutoTokenizer.from_pretrained(ck / 'tokenizer')
        tok.padding_side = 'right'
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        lm = PeftModel.from_pretrained(trunk, ck / 'adapter').eval()
        state = load_file(str(ck / norms_file))
        norms = torch.nn.ModuleList([torch.nn.LayerNorm(H) for _ in pool_layers])
        norms.load_state_dict({k[len('norms.'):]: v for k, v in state.items() if k.startswith('norms.')})
        ro = np.load(ck / 'readouts.npz')
        assert list(ro['classes']) == LABELS and list(ro['pool_layers']) == pool_layers
        dev = lm.get_input_embeddings().weight.device
        norms = norms.to(dev).float()

        def token_ids(texts):
            return [ids + [tok.eos_token_id] for ids in tok(list(texts), add_special_tokens=False, truncation=True, max_length=LORA_MAX_LEN)['input_ids']]   # these checkpoints' anchors match the tokenizer's own eos

        def features(texts, bs):
            ids = token_ids(texts)
            Z = np.zeros((len(ids), H * len(pool_layers)), dtype=np.float32)
            order = np.argsort([len(t) for t in ids], kind='stable')
            for b0 in range(0, len(order), bs):
                idx = order[b0:b0 + bs]
                batch = tok.pad({'input_ids': [ids[i] for i in idx]}, return_tensors='pt').to(dev)
                with torch.autocast('cuda', dtype=torch.float16):
                    hs = lm(**batch, output_hidden_states=True, use_cache=False).hidden_states
                m = batch['attention_mask'].unsqueeze(-1).float()
                m[:, 0] = 0   # the first token is masked out of the mean, as in training
                Z[idx] = torch.cat([nm((hs[L].float() * m.to(hs[L].device)).sum(1).to(dev) / m.sum(1).to(dev))
                                    for L, nm in zip(pool_layers, norms)], -1).cpu().numpy()
                del hs, batch
            return Z

        if (ck / anchor_file).exists():   # the checkpoint's own features for training rows 0..2: our forward pass must reproduce them
            saved = np.load(ck / anchor_file)['features'].astype(np.float32)
            ours = features(TR[:3], 3)
            rel = float(np.abs(ours - saved).mean() / (np.abs(saved).mean() + 1e-9))
            print(f'  parity with the checkpoint anchors: max abs {float(np.abs(ours - saved).max()):.4f}, mean rel {rel:.5f}', flush=True)
            if rel >= 0.02 and key == 'lora24_public':
                print(f'  SKIP {key}: features do not match the checkpoint', flush=True)
                trunk = lm.unload(); del lm; gc.collect(); continue
            assert rel < 0.02, f'{key}: our features do not match the checkpoint'
        Z = features(TE, LORA_BS)
        if key == 'lora24_public' and os.environ.get('NTL_NIGHT24_RAW'):
            np.save(os.environ['NTL_NIGHT24_RAW'], Z)
        add_member(key, softmax_rows(Z @ ro['plain_coef'].T + ro['plain_intercept']))
        if 'claude_coef' in ro.files:
            lc = Z @ ro['claude_coef'].T + ro['claude_intercept']
            GATES['claude_lora'] = lc - np.log(np.exp(lc - lc.max(1, keepdims=True)).sum(1, keepdims=True)) - lc.max(1, keepdims=True)
        if key in LORA_TWIN_W and GROUP_ROWS.any():
            Zc = group_centre_dense(Z[GROUP_ROWS], G_TE[GROUP_ROWS])
            add_member(key + '_twin', softmax_rows(Zc @ ro['center_coef'].T + ro['center_intercept']), rows=GROUP_ROWS, weight=LORA_TWIN_W[key])
        trunk = lm.unload()
        del lm, Z
        gc.collect()
        torch.cuda.empty_cache()
    del trunk
    gc.collect()
    torch.cuda.empty_cache()


if not SMOKE:
    for trunk_spec, pool_layers, checkpoints in LORA_RUNS:
        score_lora_checkpoints(trunk_spec, pool_layers, checkpoints)

# ============================================================ 6. token-attention heads + UltraBERT
class TokenAttention(torch.nn.Module if torch is not None else object):   # the module of the public training kernels (lucasboesen/name-that-llm-train-tokattn, -tokattn2)
    def __init__(self, k, h=4096):
        super().__init__()
        self.drop, self.proj, self.score, self.out = torch.nn.Dropout(0.2), torch.nn.Linear(h, 64), torch.nn.Linear(64, 1), torch.nn.Linear(128, k)

    def forward(self, x, m):
        h = torch.tanh(self.proj(self.drop(x)))
        a = torch.softmax(self.score(h).squeeze(-1).masked_fill(~m, -1e4), -1).unsqueeze(-1)
        mf = m.unsqueeze(-1).float()
        return self.out(self.drop(torch.cat([(h * mf).sum(1) / mf.sum(1).clamp(min=1), (a * h).sum(1)], -1)))


@torch.no_grad()
def token_attention_probs(texts, head_file, backbone, bs=8):
    hits = sorted(glob.glob('/kaggle/input/**/' + head_file, recursive=True))
    assert hits, f'training-kernel output holding {head_file} not attached'
    d = torch.load(hits[0], map_location='cpu')
    assert list(d['classes']) == LABELS
    decoder = d.get('decoder', True)
    path = find_checkpoint('qwen3_5') if backbone == 'qwen3_5' else find_checkpoint(None, dir_name='deberta-v3-large')
    assert path, f'backbone {backbone} not attached'
    cfg = AutoConfig.from_pretrained(path)
    if decoder:
        tc = getattr(cfg, 'text_config', None) or cfg
        tc.num_hidden_layers = d['keep']
        if getattr(tc, 'layer_types', None):
            tc.layer_types = tc.layer_types[:d['keep']]
        lm = AutoModel.from_pretrained(path, config=cfg, dtype=torch.float16, device_map='auto').eval()
    else:
        lm = AutoModel.from_pretrained(path, dtype=torch.float32).to(DEV).eval()
    tok = AutoTokenizer.from_pretrained(path)
    tok.padding_side = 'right'
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    H = d.get('hidden', 4096)
    net = TokenAttention(K, H)
    net.load_state_dict(d['state_dict'])
    dev = lm.get_input_embeddings().weight.device
    net, mu, sd = net.to(dev).eval(), d['mu'].float().to(dev), d['sd'].float().to(dev)
    ids = tok(list(texts), add_special_tokens=not decoder, truncation=True, max_length=d['max_len'])['input_ids']
    P = np.zeros((len(texts), K))
    order = np.argsort([len(t) for t in ids])
    for b0 in range(0, len(order), bs):
        idx = order[b0:b0 + bs]
        batch = tok.pad({'input_ids': [ids[i] for i in idx]}, return_tensors='pt').to(dev)
        out = lm(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'], output_hidden_states=True)
        hs = (out.hidden_states if getattr(out, 'hidden_states', None) is not None else out.language_model_outputs.hidden_states)[d['layer']]
        n = batch['attention_mask'].sum(1)
        T = max(int(n.max()) - 1, 1)
        x = torch.zeros(len(idx), T, H, device=dev)
        m = torch.zeros(len(idx), T, dtype=torch.bool, device=dev)
        for r in range(len(idx)):   # first token dropped, states rounded through fp16, as in training
            k = int(n[r]) - 1
            if k > 0:
                x[r, :k] = ((torch.nan_to_num(hs[r, 1:k + 1].to(dev).float()).half().float() - mu) / sd).clamp(-8, 8)
                m[r, :k] = True
        P[idx] = torch.softmax(net(x, m), -1).cpu().numpy()
        del hs, out, batch
    del lm
    gc.collect()
    torch.cuda.empty_cache()
    return P


@torch.no_grad()
def ultrabert_probs(texts):
    from transformers import AutoModelForSequenceClassification
    folds = None
    for root in glob.glob('/kaggle/input/**/ultrabert-380m-ntl', recursive=True):
        fs = [f for f in sorted(glob.glob(os.path.join(root, 'fold*'))) if os.path.exists(os.path.join(f, 'config.json'))
              and json.load(open(os.path.join(f, 'config.json'))).get('model_type') == 'ultrabert']
        if len(fs) == 5:
            folds = fs
            break
    assert folds, 'ultrabert-380m-ntl (5 folds) not attached'
    tok = AutoTokenizer.from_pretrained(folds[0], trust_remote_code=True)
    ids = tok(texts, truncation=True, max_length=512)['input_ids']
    order = np.argsort([len(e) for e in ids])
    P = np.zeros((len(texts), K))
    for fp in folds:
        em = AutoModelForSequenceClassification.from_pretrained(fp, trust_remote_code=True, dtype=torch.float16, attn_implementation='sdpa').to(DEV).eval()
        cols = [LAB2I[em.config.id2label[i]] for i in range(len(em.config.id2label))]
        for b0 in range(0, len(order), 32):
            idx = order[b0:b0 + 32]
            b = tok.pad({'input_ids': [ids[j] for j in idx]}, return_tensors='pt').to(DEV)
            P[np.ix_(idx, cols)] += em(**b).logits.float().softmax(-1).cpu().numpy() / len(folds)
        del em
        gc.collect()
        torch.cuda.empty_cache()
    return P


if not SMOKE:
    # Optional community heads are enabled when both the trained head and matching backbone are attached.
    _available_tokattn = []
    for _key, _head_file, _backbone in TOKATTN_HEADS:
        _head_hits = glob.glob('/kaggle/input/**/' + _head_file, recursive=True)
        _base_hit = find_checkpoint('qwen3_5') if _backbone == 'qwen3_5' else find_checkpoint(None, dir_name='deberta-v3-large')
        if _head_hits and _base_hit:
            _available_tokattn.append((_key, _head_file, _backbone))
        else:
            print(f'  OPTIONAL HEAD SKIPPED {_key}: head={bool(_head_hits)} backbone={bool(_base_hit)}', flush=True)
    TOKATTN_HEADS = _available_tokattn
    for key, head_file, backbone in TOKATTN_HEADS:
        if TINY:   # in-sample check on training rows the head was fitted on: must be high, or the checkpoint is not being reproduced
            acc = float(np.mean(token_attention_probs(TR[:40], head_file, backbone).argmax(1) == y[:40]))
            print(f'  {key} in-sample check on 40 training rows: acc {acc:.2f}', flush=True)
            assert acc > 0.5, f'{key} does not reproduce its training fit'
        add_member(key, token_attention_probs(TE, head_file, backbone))
    add_member('ultrabert', ultrabert_probs(TE))


def _np_logsoftmax(z):
    from scipy.special import logsumexp as _lse_np
    z = np.asarray(z, dtype=np.float64)
    return z - _lse_np(z, axis=1, keepdims=True)

def _apply_conditional_family_gate(gate_key, family, lam, scores_are_logprob=False):
    '''Use the already-trained family gate only to rerank siblings, preserving total family mass.
    Confidence-aware interpolation avoids overriding high-confidence cross-family decisions.'''
    score = GATES.get(gate_key)
    if score is None:
        print(f'  family gate skipped {gate_key}: no fitted scores', flush=True)
        return
    idx = FAMILY_IDX[family]
    score = np.asarray(score, dtype=np.float64)
    if score.shape != (len(TE), len(idx)):
        print(f'  family gate skipped {gate_key}: shape {score.shape} expected {(len(TE), len(idx))}', flush=True)
        return
    if scores_are_logprob:
        gate_lp = _np_logsoftmax(score)
    else:
        score = score - score.mean(axis=1, keepdims=True)
        scale = np.sqrt(np.mean(score * score, axis=1, keepdims=True))
        score = score / np.maximum(scale, 0.25)
        gate_lp = _np_logsoftmax(score)
    base = LOGIT[:, idx].copy()
    base_lp = _np_logsoftmax(base)
    base_p = np.exp(base_lp)
    sp = np.sort(base_p, axis=1)
    margin = sp[:, -1] - sp[:, -2]
    confidence_gate = np.clip((0.45 - margin) / 0.45, 0.0, 1.0)
    strength = min(0.45, float(lam) / (1.0 + float(lam))) * confidence_gate
    mixed_lp = _np_logsoftmax((1.0 - strength[:, None]) * base_lp + strength[:, None] * gate_lp)
    # Preserve the ensemble's probability mass assigned to this family; only adjust within-family identity.
    from scipy.special import logsumexp as _lse_np
    family_mass = _lse_np(base, axis=1)
    LOGIT[:, idx] = family_mass[:, None] + mixed_lp
    print(f'  family gate applied {gate_key}: mean strength={strength.mean():.4f}, active rows={int((strength > 0).sum())}', flush=True)

# These gates were fitted above but were previously never added to LOGIT. Apply them conservatively.
if not SMOKE:
    _apply_conditional_family_gate('claude', 'claude', GATE_LAM['claude_style'])
    _apply_conditional_family_gate('claude_lora', 'claude', GATE_LAM['claude_lora'], scores_are_logprob=True)
    for _fam in ('gpt', 'gemini', 'glm'):
        _apply_conditional_family_gate(_fam, _fam, GATE_LAM['family'])

np.save('/kaggle/working/comp_logit.npy', LOGIT.astype(np.float32))
json.dump({'labels': LABELS, 'ids': test['id'].astype(str).tolist(), 'seconds': time.time() - T0,
           'family_gates_applied': bool(not SMOKE)}, open('/kaggle/working/comp_meta.json', 'w'))
print(f'COMP STACK DONE {time.time() - T0:.0f}s', flush=True)

In [ ]:
import time as _t, os, subprocess, sys, shutil, numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
_c0 = _t.time()
os.environ['NTL_DROP_Q35'] = '1'
os.environ['NTL_NIGHT24_RAW'] = '/tmp/night24_raw.npy'

def _compute_topic_groups(texts):
    try:
        v = TfidfVectorizer(sublinear_tf=True, stop_words='english', min_df=2, max_df=0.05)
        Xc = v.fit_transform(texts)
        S = (Xc @ Xc.T).toarray().astype(np.float32)
        n = len(S); np.fill_diagonal(S, -np.inf)
        size = np.ones(n, np.int32); alive = np.ones(n, bool); lab = np.arange(n)
        while True:
            ok = (size[:, None] + size[None, :] <= 18) & alive[:, None] & alive[None, :]
            M = np.where(ok, S, -np.inf); k = int(np.argmax(M)); i, j = divmod(k, n)
            if not np.isfinite(M[i, j]) or M[i, j] < 0.15: break
            new = (size[i] * S[i] + size[j] * S[j]) / (size[i] + size[j])
            S[i] = new; S[:, i] = new; S[i, i] = -np.inf; S[j] = -np.inf; S[:, j] = -np.inf
            size[i] += size[j]; alive[j] = False; lab[lab == j] = i
        return lab
    except Exception:
        return np.arange(len(texts), dtype=np.int64)

if not FAST_COMMIT:
    subprocess.run([sys.executable, 'comp_stack.py'], check=True)
    if os.path.exists('/tmp/night24_raw.npy'):
        subprocess.run([sys.executable, 'night24_helper.py', '/tmp/night24_raw.npy', os.path.join(DATA, 'test.csv'), '/kaggle/working/night24_helper.npz'], check=True)
    # Run Full-Text 64-Layer Qwen3.8-27B GSQ3 across both T4 GPUs
    _g0 = _t.time()
    try:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--disable-pip-version-check', '--target', '/tmp/gsq-python', *GSQ_WHEELS], check=True)
        _df_te = pd.read_csv(os.path.join(DATA, 'test.csv'), dtype={'id': str})
        _df_te['text'] = _df_te['text'].fillna('').astype(str)
        _df_te.to_parquet('/tmp/gsq_query.parquet', index=False)
        np.save('/tmp/gsq_groups.npy', _compute_topic_groups(_df_te['text'].tolist()), allow_pickle=False)
        shutil.copytree(GSQ_BASE_SRC, '/tmp/gsq3-base', dirs_exist_ok=True)
        _genv = dict(os.environ, PYTHONPATH='/tmp/gsq-python:/kaggle/working/gsq', HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', TOKENIZERS_PARALLELISM='false')
        subprocess.run([
            sys.executable, '-u', '/kaggle/working/gsq/infer.py',
            '--base', '/tmp/gsq3-base',
            '--final', GSQ_FINAL,
            '--metadata', GSQ_META,
            '--complete', GSQ_COMP,
            '--input', '/tmp/gsq_query.parquet',
            '--groups', '/tmp/gsq_groups.npy',
            '--output', '/kaggle/working/gsq27b.npz',
            '--work-dir', '/tmp/gsq-workers',
            '--batch-size', '1'
        ], env=_genv, check=True, timeout=3600)
        print('GSQ 27B finished in', round(_t.time() - _g0), 's', flush=True)
    except Exception as _ge:
        print('GSQ 27B skipped due to error/timeout:', repr(_ge), flush=True)
    finally:
        shutil.rmtree('/tmp/gsq3-base', ignore_errors=True)
        shutil.rmtree('/tmp/gsq-workers', ignore_errors=True)
else:
    _src = open('comp_stack.py').read()
    _ns = {'glob': glob, 'os': os, 'json': json}
    exec(_src[_src.index('def find_checkpoint('):_src.index('def load_truncated_decoder(')], _ns)
    for _mt, _h in [('qwen3', 4096), ('qwen3', 2560), ('qwen2', 3584), ('glm4', 4096)]:
        _ck_found = _ns['find_checkpoint'](_mt, _h)
        print('find_checkpoint', _mt, _h, '->', _ck_found, flush=True)
        assert _ck_found, (_mt, _h)
    subprocess.run([sys.executable, 'night24_helper.py', 'random', os.path.join(DATA, 'test.csv'), '/tmp/night24_preview.npz'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--disable-pip-version-check', '--target', '/tmp/gsq-python', *GSQ_WHEELS], check=True)
    _genv = dict(os.environ, PYTHONPATH='/tmp/gsq-python:/kaggle/working/gsq')
    subprocess.run([sys.executable, '-c', 'import gsq_infer_model, gsq_infer_layers, delta_turing, adapter; print("GSQ 27B modules imported OK")'], env=_genv, check=True)
COMP_SEC = _t.time() - _c0
print('comp + gsq stack wall time', round(COMP_SEC), flush=True)

In [ ]:
import pandas as pd, subprocess, sys, json, os

L = [2, 4, 6, 8, 12, 16, 20, 24]
MODELS = [
  {"name": "q14v5", "type": "l24", "path": B14, "adapters": AQ14V5, "keep": 27, "layers": L, "w": 3.0, "bs": 8, "max_len": 384, "twin": 0.35, "gauss": 0.70},
  {"name": "q8v5", "type": "l24", "path": B8, "adapters": AQ8V5, "keep": 24, "layers": L, "w": 2.0, "bs": 16, "max_len": 384, "twin": 0.35, "gauss": 0.44, "dp": True},
  {"name": "ll8v5", "type": "l24", "path": BL, "adapters": ALL8V5, "keep": 24, "layers": L, "w": 1.0, "bs": 16, "max_len": 384, "twin": 0.35, "dp": True},
  {"name": "n14b", "type": "l24", "path": B14, "adapters": [N14B], "keep": N14BC["keep"], "layers": N14BC["layers"], "w": 2.5, "bs": 8, "max_len": N14BC["max_len"], "twin": 0.35, "no_final_norm": True, "eos_id": N14BC["eos_id"]},
  {"name": "q25_14v5", "type": "l24", "path": B25_14, "adapters": AQ25_14V5, "keep": 32, "layers": [3, 5, 8, 11, 16, 21, 27, 32], "w": 1.6, "bs": 8, "max_len": 384, "twin": 0.35},
  {"name": "q25_14", "type": "l24", "path": B25_14, "adapters": [Q25_14], "keep": Q25_14C["keep"], "layers": Q25_14C["layers"], "w": 0.6, "bs": 8, "max_len": Q25_14C["max_len"], "twin": 0.35, "no_final_norm": True, "eos_id": Q25_14C["eos_id"]},
  {"name": "deb", "type": "enc", "ckpt": DEB, "w": 1.0},
]

ATLAS_BUDGET = int(8.3 * 3600 - COMP_SEC - 600)
MEMBER_DEADLINE = max(3600, ATLAS_BUDGET - 4800)

cmd = [
    sys.executable, 'infer.py',
    '--data', str(DATA),
    '--models', json.dumps(MODELS),
    '--perpassage', '1',
    '--extra', str(EXTRA),
    '--extra_w', '0.5',
    '--tfidf_w', '0.25',
    '--distort_w', '0.10',
    '--svc_w', '1.5',
    '--pool', str(POOL),
    '--adapt_w', '1.0',
    '--adapt_thr', '0.15',
    '--sinkhorn', '0.45',
    '--sinkhorn_target_mix', '0.35',
    '--post', '0',
    '--atlas_dir', str(ATLAS),
    '--atlas_base', str(B8),
    '--atlas_w', '0.20',
    '--atlas_margin', '0.3',
    '--atlas_bs', '4',
    '--atlas_chunk', '3',
    '--atlas_budget', str(max(180, ATLAS_BUDGET)),
    '--member_deadline', str(MEMBER_DEADLINE),
    '--comp_path', '/kaggle/working/comp_logit.npy',
    '--comp_w', '0.15',
    '--gsq_path', '/kaggle/working/gsq27b.npz',
    '--gsq_lp_w', '1.5',
    '--gsq_prob_w', '0.05',
    '--helper_path', '/kaggle/working/night24_helper.npz',
    '--helper_w', '0.04',
    '--out', '/kaggle/working/submission.parquet',
    '--cache', '/kaggle/working/cache'
]
print('RUNNING MAIN INFERENCE', flush=True)
if FAST_COMMIT:
    _test_ph = pd.read_csv(os.path.join(DATA, 'test.csv'))
    pd.DataFrame({'id': _test_ph.id, 'model': 'gpt-5.5'}).to_parquet('/kaggle/working/submission.parquet', index=False)
else:
    subprocess.run(cmd, check=True)
submission = pd.read_parquet('/kaggle/working/submission.parquet')
assert len(submission) == len(pd.read_csv(os.path.join(DATA, 'test.csv')))
assert set(submission.columns) == {'id', 'model'} and submission.id.is_unique and submission.model.notna().all()
print(submission.head(), flush=True)